# ◈ STARK INDUSTRIES · NEURAL TRANSLATION ARRAY
### `v10.0` · Gemma3:27b · Story State Engine v5.0 · 2-Pass · ToneGuard Pro · AuthorDNA · T4 Session Guard

```
  ╔══════════════════════════════════════════════════════════════════════════╗
  ║  J.A.R.V.I.S  —  Just A Rather Very Intelligent System                  ║
  ║  Hinglish Literary Translation Engine v10.0                              ║
  ║  Model: gemma3:27b  ·  2-Pass  ·  T4 (15GB)  ·  Public Domain Edition  ║
  ╚══════════════════════════════════════════════════════════════════════════╝
```

| Step | Cell | Mission |
|------|------|---------|
| **①** | 2 | Install dependencies |
| **②** | 4–5 | Boot Ollama · Pull gemma3:27b |
| **③** | 7 | Upload source `.txt` |
| **④** | 9–10 | Configure parameters |
| **⑤** | 12 | Story State Engine v5.0 |
| **⑥** | 14 | Load Translation Engine v10.0 |
| **⑦** | 16 | **⚡ Execute** |
| **⑧** | 18 | Download output |

> **v10.0 Upgrades over v9.2**  
> • **AuthorDNA** — Pre-built profiles for Kafka, Austen, Doyle, Tagore, Stoker, Tolstoy, Chekhov, etc.  
> • **Richer Few-Shots** — 8 genre-specific examples embedded in Step2 prompt (Hinglish voice calibrated)  
> • **Hinglish Voice Guide** — Researched tone: OTT-India register (Made in Heaven, Little Things style)  
> • **T4 Session Guard** — Real-time session timer, adaptive chunking, auto-checkpoint every chunk  
> • **Anti-Hallucination Layer** — Zero-fabrication rule, source-anchoring in Step1 & Step2  
> • **Story State v5.0** — Rolling 3-chunk context summary, author tone locked per book  
> • **Background I/O** — File writes off main thread, no translation blocking  
> • **Context Compression** — Context block capped at 800 chars max — model never overloaded


## ⚡ Step 1 — Install Dependencies
Run once per Colab session.

In [1]:
!pip install -q ollama ipywidgets
import torch
from IPython.display import display, HTML
print(f'✅ Dependencies ready | CUDA: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
display(HTML('''
<div style="background:#070710;border:2px solid #c0392b;border-radius:8px;
            padding:12px 18px;font-family:'Courier New',monospace;margin-top:10px;">
  <div style="color:#c0392b;font-size:1.1em;font-weight:bold;letter-spacing:2px;">[ STARK INDUSTRIES — JARVIS v10.0 ]</div>
  <div style="color:#4CAF50;font-size:0.85em;margin-top:4px;">ollama · ipywidgets · torch ready</div>
  <div style="color:#FFD700;font-size:0.78em;margin-top:3px;">AuthorDNA · ToneGuard Pro · T4 Session Guard · Anti-Hallucination Layer</div>
</div>
'''))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.0 MB/s eta 0:00:00
✅ Dependencies ready | CUDA: True | Device: Tesla T4


## 🦙 Step 2a — Boot Ollama Server
Run Cell 4, then Cell 5.

> ⚠️ T4 = 15 GB VRAM. gemma3:27b Q4_K_M needs ~17 GB — Ollama uses CPU offload for extra layers. Expect ~60–100s/chunk (2-pass). For 40K words (~91 chunks): **2.5–3 hours**.

In [2]:
import subprocess, time, os
from IPython.display import display, HTML

display(HTML('<div style="background:linear-gradient(135deg,#0a0a0f,#1a0505);border:2px solid #c0392b;'
            'border-radius:8px;padding:14px 18px;font-family:Courier New,monospace;">'
            '<div style="color:#c0392b;font-size:1.2em;font-weight:bold;letter-spacing:3px;">◈ OLLAMA BOOT SEQUENCE</div>'
            '<div style="color:#FFD700;font-size:0.82em;margin-top:4px;">gemma3:27b | v10.0 | T4 Session Guard active</div></div>'))

!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

print('\n🚀 Starting Ollama server...')
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
os.environ['OLLAMA_GPU_OVERHEAD'] = '512000000'
subprocess.Popen(['/usr/local/bin/ollama', 'serve'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)

try:
    import ollama; ollama.list()
    display(HTML('<div style="background:#0a0f0a;border:2px solid #4CAF50;border-radius:6px;'
                'padding:10px 18px;font-family:Courier New,monospace;margin-top:8px;">'
                '<span style="color:#4CAF50;font-weight:bold;">[OK] ARC REACTOR STABLE — Ollama server operational.</span><br>'
                '<span style="color:#888;font-size:0.82em;">Run Cell 5 to pull gemma3:27b (~17 GB — takes 10-25 min).</span></div>'))
except Exception as e:
    print(f'⚠️ Server may still be starting: {e} — wait 5s and re-run.')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

🚀 Starting Ollama server...


In [3]:
import ollama

MODEL_NAME = 'gemma3:27b'
print(f'📥 Pulling {MODEL_NAME} (Q4_K_M ~17 GB) — go make chai ☕')

try:
    current_digest = ''
    for progress in ollama.pull(MODEL_NAME, stream=True):
        digest = progress.get('digest', '')
        if digest != current_digest and current_digest: print()
        current_digest = digest
        status = progress.get('status', '')
        if 'completed' in progress and 'total' in progress:
            pct = (progress['completed'] / progress['total'] * 100) if progress['total'] else 0
            bar = '█' * int(pct / 2) + '░' * (50 - int(pct / 2))
            print(f'\r   [{bar}] {pct:.1f}%', end='', flush=True)
        else:
            print(f'\r   {status}', end='', flush=True)
    print(f'\n\n✅ {MODEL_NAME} ready!')
    for m in ollama.list().get('models', []):
        print(f"   • {m.get('name')} ({m.get('size',0)/(1024**3):.2f} GB)")
except Exception as e:
    print(f'\n❌ Pull failed: {e}\n   Make sure Ollama server is running (Cell 4).')

📥 Pulling gemma3:27b (Q4_K_M ~17 GB) — go make chai ☕
   [██████████████████████████████████████████████████] 100.0%
   [██████████████████████████████████████████████████] 100.0%
   [██████████████████████████████████████████████████] 100.0%
   [██████████████████████████████████████████████████] 100.0%
   [██████████████████████████████████████████████████] 100.0%
   success

✅ gemma3:27b ready!
   • None (16.20 GB)


## 📤 Step 3 — Upload Source File
Upload your `.txt` book file (any language — will be pre-processed).

In [4]:
from IPython.display import display, HTML
from google.colab import files
import re

display(HTML('<div style="background:linear-gradient(135deg,#0a0a0f,#1a0505);border:2px solid #c0392b;'
            'border-radius:8px;padding:14px 18px;font-family:Courier New,monospace;margin-bottom:8px;">'
            '<div style="color:#c0392b;font-size:1.15em;font-weight:bold;letter-spacing:2px;">◈ FILE UPLINK</div>'
            '<div style="color:#FFD700;font-size:0.85em;margin-top:4px;">Select your .txt file (English or pre-translated chapter).</div></div>'))

uploaded = files.upload()
UPLOADED_FILE = list(uploaded.keys())[0]

with open(UPLOADED_FILE, 'r', encoding='utf-8') as f:
    _raw = f.read()

# Auto-strip pipeline headers from previous runs
_cleaned = re.sub(
    r'^TRANSLATED TO ENGLISH\s*[=\-]{10,}.*?[=\-]{10,}+','', _raw, flags=re.DOTALL
).strip()
if _cleaned != _raw:
    with open(UPLOADED_FILE, 'w', encoding='utf-8') as f:
        f.write(_cleaned)
    print('[OK] Pipeline header stripped.')

_words = len(_cleaned.split())
print(f'\n✅ File ready: {UPLOADED_FILE}')
print(f'   {_words:,} words | {len(_cleaned):,} chars')
print(f'   Preview: {_cleaned[:200].strip()!r}...')

Saving kapitel_02_I.txt to kapitel_02_I.txt
[OK] Pipeline header stripped.

✅ File ready: kapitel_02_I.txt
   1,156 words | 6,576 chars
   Preview: 'Gregor was now shut off from his mother, who, through his fault, was perhaps close to death; he wasn’t allowed to open the door, or he would scare away his sister, who had to stay with their mother; h'...


## ⚙️ Step 4 — Configure Parameters

| Parameter | Default | Notes |
|-----------|---------|-------|
| Chunk size | **350 words** | Optimized for T4 — good quality/speed balance |
| Overlap | **80 words** | Ensures continuity at chunk boundaries |
| num_ctx | **8192** | Max context window — keep at 8192 for T4 |

> **T4 Session Guard**: The engine tracks elapsed time and warns if session timeout is approaching.

In [5]:
import ipywidgets as widgets
from IPython.display import display, HTML

TIER_OPTIONS = {
    'ADVANCED  — Full Millennial Hinglish (2-pass, recommended)': 'ADVANCED',
    'INTERMEDIATE  — Balanced (single-pass)': 'INTERMEDIATE',
    'BASIC  — Fast, clean Hindi (single-pass)': 'BASIC',
}
tier_dropdown = widgets.Dropdown(
    options=list(TIER_OPTIONS.keys()),
    value='ADVANCED  — Full Millennial Hinglish (2-pass, recommended)',
    description='Quality Tier:', style={'description_width':'initial'},
    layout=widgets.Layout(width='520px')
)
chunk_slider = widgets.IntSlider(
    value=350, min=150, max=550, step=25,
    description='Chunk size (words):', style={'description_width':'initial'},
    layout=widgets.Layout(width='520px')
)
overlap_slider = widgets.IntSlider(
    value=80, min=0, max=150, step=10,
    description='Overlap (words):', style={'description_width':'initial'},
    layout=widgets.Layout(width='520px')
)
num_ctx_slider = widgets.IntSlider(
    value=8192, min=4096, max=12288, step=1024,
    description='num_ctx (tokens):', style={'description_width':'initial'},
    layout=widgets.Layout(width='520px')
)
session_slider = widgets.IntSlider(
    value=270, min=60, max=360, step=10,
    description='Session budget (min):', style={'description_width':'initial'},
    layout=widgets.Layout(width='520px')
)
display(HTML('<div style="background:#0a0a0f;border:2px solid #c0392b;border-radius:8px;'
            'padding:12px 18px;font-family:Courier New,monospace;margin-bottom:10px;">'
            '<div style="color:#c0392b;font-weight:bold;letter-spacing:2px;">◈ MISSION PARAMETERS — JARVIS v10.0</div>'
            '<div style="color:#888;font-size:0.78em;margin-top:4px;">Session Guard | AuthorDNA | ToneGuard Pro | Anti-Hallucination</div>'
            '</div>'))
display(tier_dropdown, chunk_slider, overlap_slider, num_ctx_slider, session_slider)
print('\n💡 Adjust then run Cell 10 to lock config.')

Dropdown(description='Quality Tier:', layout=Layout(width='520px'), options=('ADVANCED  — Full Millennial Hing…

IntSlider(value=350, description='Chunk size (words):', layout=Layout(width='520px'), max=550, min=150, step=2…

IntSlider(value=80, description='Overlap (words):', layout=Layout(width='520px'), max=150, step=10, style=Slid…

IntSlider(value=8192, description='num_ctx (tokens):', layout=Layout(width='520px'), max=12288, min=4096, step…

IntSlider(value=270, description='Session budget (min):', layout=Layout(width='520px'), max=360, min=60, step=…


💡 Adjust then run Cell 10 to lock config.


In [6]:
import os
MODEL          = 'gemma3:27b'
CHUNK_SIZE     = chunk_slider.value
OVERLAP_WORDS  = overlap_slider.value
TRANSLATION_TIER = TIER_OPTIONS[tier_dropdown.value]
NUM_CTX        = num_ctx_slider.value
SESSION_BUDGET = session_slider.value * 60   # in seconds
OUTPUT_DIR     = './translation_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Configuration locked:')
print(f'   🤖 Model          : {MODEL}')
print(f'   🎯 Quality Tier   : {TRANSLATION_TIER}')
print(f'   📦 Chunk Size     : {CHUNK_SIZE} words')
print(f'   🔀 Overlap        : {OVERLAP_WORDS} words')
print(f'   🧠 num_ctx        : {NUM_CTX} tokens')
print(f'   ⏱️  Session Budget : {SESSION_BUDGET//60} min')
print()
print('ℹ️  Ollama Parameters (fixed for 2-pass):')
print('   Step1: temp=0.15  top_k=20  top_p=0.85  (faithful base)')
print('   Step2: temp=0.65  top_k=40  top_p=0.90  (Hinglish style)')

✅ Configuration locked:
   🤖 Model          : gemma3:27b
   🎯 Quality Tier   : ADVANCED
   📦 Chunk Size     : 350 words
   🔀 Overlap        : 80 words
   🧠 num_ctx        : 8192 tokens
   ⏱️  Session Budget : 270 min

ℹ️  Ollama Parameters (fixed for 2-pass):
   Step1: temp=0.15  top_k=20  top_p=0.85  (faithful base)
   Step2: temp=0.65  top_k=40  top_p=0.90  (Hinglish style)


## 🧠 Step 5 — Story State Engine v5.0

**v5.0 Upgrades:**
- ✅ `AuthorDNA` — Pre-built profiles for 12+ classic authors (auto-loaded from BOOK_PROFILE)
- ✅ Rolling 3-chunk event summary (not just 1) — better long-book continuity
- ✅ `context_compression()` — hard cap at 800 chars — model never overloaded
- ✅ `tone_anchor` per book — locks the register (formal/lyrical/sardonic etc.)
- ✅ Resume-from-chunk (not just chapter) — granular mid-run recovery

**Manual setup (optional, before Cell 16):**
```python
add_vocab_entry('the game is afoot', 'khel shuru ho gaya')
update_character('Holmes', role='eccentric detective', address='aap',
                 speech_style='calm, precise, analytical — never informal')
```

In [7]:
import re, json, os
from collections import Counter
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════════
# STORY STATE ENGINE v5.0  — JARVIS v10.0
# Upgrades: AuthorDNA · Rolling 3-chunk context · context_compression()
#           tone_anchor · resume-from-chunk
# ═══════════════════════════════════════════════════════════════════════

# STORY_STATE = {
#     'book_title':            '',
#     'genre':                 '',
#     'author':                '',
#     'author_dna':            '',       # v5.0: author tone fingerprint
#     'tone_anchor':           '',       # v5.0: locked register for entire book
#     'characters':            {},
#     'character_speech_styles': {},
#     'current_setting':       '',
#     'dominant_tone':         'DAILY_LIFE',
#     'established_vocab':     {},
#     'story_so_far':          [],       # v5.0: rolling 3-entry summary
#     'chunk_count':           0,
#     'total_chunks':          0,
#     'resume_from_chunk':     0,        # v5.0: granular resume
# }

STORY_STATE = {

    # ─── BOOK IDENTITY ────────────────────────────────────────────────
    'title': 'The Metamorphosis (Die Verwandlung) — Chapter 2',

    # ─── AUTHOR DNA KEY ───────────────────────────────────────────────
    'author_key': 'kafka',

    # ─── GENRE ────────────────────────────────────────────────────────
    'genre': 'literary fiction / absurdist',

    # ─── CHARACTERS ───────────────────────────────────────────────────
    'characters': [
        (
            'Gregor',
            'transformed protagonist',
            (
                'Now fully adapted to insect body — crawls walls and ceiling for pleasure. '
                'Hides under sofa with a sheet. Protects framed picture of lady in furs. '
                'Likes rotting scraps, hates fresh food. Apple lodged in back. '
                'Eyesight failing. Listens at doors. Accepts everything with flat resignation.'
            ),
            'woh',
            'anxious internal monologue, long run-on sentences, self-rationalizing — never dramatic'
        ),
        (
            'Grete',
            'younger sister turned caretaker',
            (
                'Self-appointed expert on Gregor. Cleans room, tests food, strips furniture. '
                'Childish authority. First direct address to Gregor: "You, Gregor!" — fist raised.'
            ),
            'tum',
            'practical, efficient; brief and cold when addressing Gregor directly'
        ),
        (
            'Father',
            'head of household — newly authoritative',
            (
                'Wears bank uniform daily. Throws apples — one lodges in Gregor\'s back. '
                'Misreads every situation. Completely changed from the dressing-gown man of before.'
            ),
            'aap',
            'commanding, short sentences, no explanations — orders and judgments only'
        ),
        (
            'Mother',
            'frail, asthmatic matriarch',
            (
                'Asthmatic. Enters room to move furniture — argues to keep it unchanged. '
                'Faints at the wallpaper stain. Runs out in nightgown pleading for Gregor.'
            ),
            'aap',
            'gentle, whispering, pleading — terrified but fiercely protective'
        ),
    ],

    # ─── EXTRA VOCAB ──────────────────────────────────────────────────
    'extra_vocab': {
        'the manager':          'Manager sahab',
        'transformation':       'tabdeeli',
        'vermin':               'keeda-makoda',
        'father':               'Papa',
        'mother':               'Maa',
        'sister':               'behen',
        'debt':                 'karza',
        'office':               'office',
        'living room':          'baithak',
        'front room':           'aagla kamra',
        'ceiling':              'chhat',
        'sofa':                 'sofa',
        'armchair':             'aaraam kursi',
        'sheet':                'chadar',
        'chest of drawers':     'badi almari',
        'desk':                 'desk',
        'sideboard':            'sideboard',
        'fruit bowl':           'phalon ki tokri',
        'apple':                'seb',
        'uniform':              'vardi',
        'antennae':             'moochen',
        'the picture':          'woh tasveer',
        'lady in furs':         'khaal wale kapdon wali aurat',
        'wallpaper':            'deewaar ka kaagaz',
        'nightgown':            'raat ka libaas',
        'tonic':                'dawai',
        'asthma':               'dama',
        'safe':                 'tijori',
        'conservatory':         'music school',
        'commission':           'commission',
        'traveling salesman':   'traveling salesman',
        'newspaper':            'akhbaar',
        'the maid':             'kaam waali bai',
        'bank':                 'bank',
    },

    # ─── TONE RULES ───────────────────────────────────────────────────
    'tone_rules': {
        'default': 'flat, neutral, matter-of-fact — dead-eyed Kafka bureaucratic register',
        'avoid': [
            'dramatic exclamations',
            'horror-genre language',
            'sympathy-seeking narration',
            'sarcasm or attitude',
            'street slang or GenZ filler',
            'emotional amplification of any kind',
        ],
        'extra': (
            'Festering apple wound = logistical problem, not body horror. '
            'Family financial meeting = routine admin, not despair. '
            'Father throwing apples = mechanical event, not violence. '
            'Mother collapsing = ordinary, not operatic. '
            'The absurd and the mundane share the exact same flat, tired tone.'
        ),
    },

    # ─── RESUME ───────────────────────────────────────────────────────
    'resume_from': '',

}

# ── AuthorDNA Profiles ─────────────────────────────────────────────────
# Each profile: (tone_anchor, author_dna_description, genre, extra_vocab_hints)
# These are pre-researched based on how Indian millennials best receive these authors.
AUTHOR_DNA = {
    'kafka': (
        'unsettling, flat, bureaucratic',
        ('Franz Kafka style: Describe impossible or absurd things in completely matter-of-fact, '
         'emotionless language. The horror comes from the FLATNESS — not dramatic reactions. '
         'Characters accept the unacceptable as routine. Every sentence should feel like a '
         'government notice being read aloud. No shock, no exclamation, just cold observation. '
         'Hinglish must preserve this dead-eyed bureaucratic register.'),
        'literary fiction',
        {'transformation': 'tabdeeli', 'vermin': 'keeda-makoda', 'chief clerk': 'chief clerk sahab',
         'office': 'office', 'debt': 'karza', 'father': 'Papa', 'mother': 'Maa', 'sister': 'behen'}
    ),
    'austen': (
        'witty, ironic, socially observant',
        ('Jane Austen style: Dry wit and gentle irony about marriage, class, and social pretense. '
         'The humor is in UNDERSTATEMENT — the narrator sees through everyone\'s pretensions. '
         'Dialogue reveals character. Formal social settings, sharp observations about reputation. '
         'Hinglish must carry the irony — a raised eyebrow in prose. '
         'Never let the wit become sarcasm. It\'s knowing, not cutting.'),
        'romance/social',
        {'estate': 'haveli', 'fortune': 'daulat', 'society': 'samaj ke log',
         'governess': 'ghar ki teacher', 'the ball': 'dance party', 'proposal': 'rishta',
         'marriage': 'shaadi', 'reputation': 'izzat', 'the neighbourhood': 'aas-paas ke log'}
    ),
    'doyle': (
        'confident, precise, Victorian-formal',
        ('Arthur Conan Doyle (Sherlock Holmes) style: Holmes speaks with absolute analytical '
         'confidence — never casual, never uncertain. Watson narrates with warmth and admiration. '
         'The atmosphere is foggy Victorian London — gas lamps, hansom cabs, drawing rooms. '
         'Deduction scenes are methodical: one observation leads to the next. '
         'Hinglish: Holmes = "aap" register, formal, precise. Watson = warm narrator. '
         'Holmes NEVER says anything casual. His Hinglish sounds like a composed professional.'),
        'mystery/detective',
        {'deduction': 'deduction', 'elementary': 'seedhi baat hai', 'the game is afoot': 'khel shuru ho gaya',
         'inspector': 'inspector sahab', 'clue': 'suraag', 'the case': 'yeh case',
         'hansom cab': 'buggy', 'telegram': 'taaar', 'revolver': 'revolver', 'Baker Street': 'Baker Street'}
    ),
    'tagore': (
        'lyrical, philosophical, deeply emotional',
        ('Rabindranath Tagore style: Rich inner life, nature imagery, spiritual yearning, '
         'complex human relationships. Sentences breathe — long and lyrical, then suddenly short '
         'and piercing. The emotional register is ELEVATED but never melodramatic. '
         'Love, loss, freedom, identity — felt deeply. Bengali cadence: thoughts arrive slowly, '
         'building to a realization. Hinglish must honour the lyricism — flowing sentences, '
         'poetic imagery kept intact. No flippancy. Even simple moments carry weight.'),
        'literary fiction',
        {'ashram': 'ashram', 'river': 'nadi', 'homeland': 'apna desh', 'freedom': 'azaadi',
         'devotion': 'bhakti', 'the heart': 'mann', 'beloved': 'priya', 'sorrow': 'dard'}
    ),
    'stoker': (
        'atmospheric, Gothic, dread-building',
        ('Bram Stoker style: Gothic atmosphere — crumbling castles, howling winds, supernatural dread. '
         'Told through diary entries and letters — personal, immediate fear. '
         'Horror builds slowly: something wrong before anything explicitly terrible happens. '
         'Victorian formality + creeping unease. Characters are rational people encountering '
         'the irrational. Hinglish: preserve the formal diary-entry register. '
         'Let dread build through DESCRIPTION, not exclamation. Cold and clinical about the horrifying.'),
        'gothic/horror',
        {'the Count': 'Count sahab', 'castle': 'woh purani haveli', 'wolves': 'bhediye',
         'blood': 'khoon', 'the undead': 'woh marne ke baad bhi jeene wale', 'stake': 'khoonti',
         'journal': 'diary', 'the carriage': 'baghgi', 'Transylvania': 'Transylvania'}
    ),
    'tolstoy': (
        'epic, morally weighty, socially panoramic',
        ('Leo Tolstoy style: Epic scale — aristocratic Russian society, war, spiritual searching. '
         'Characters wrestle with profound moral questions. Narration is omniscient and unhurried. '
         'Small domestic scenes carry enormous weight. Inner monologue is rich and honest. '
         'Hinglish: maintain the gravitas. No casualness in serious moments. '
         'Russian names and titles kept as-is. The moral weight must come through.'),
        'literary fiction',
        {'prince': 'Shahzada', 'countess': 'Countess sahiba', 'estate': 'zamindari',
         'ball': 'mehfil', 'regiment': 'regiment', 'serfs': 'kaamgar log', 'samovar': 'chai ki ketli'}
    ),
    'dostoevsky': (
        'psychologically intense, confessional, anguished',
        ('Fyodor Dostoevsky style: Characters on the edge — psychologically raw, morally tormented. '
         'Interior monologue that spirals and contradicts itself. Desperate poverty, '
         'religious guilt, underground resentment. Sentences accelerate with the character\'s anxiety. '
         'Hinglish: let the inner voice be unstable, fragmented. A character thinking in circles. '
         'The anguish must come through — but without melodrama. Raw, not theatrical.'),
        'literary fiction',
        {'conscience': 'zameer', 'suffering': 'takleef', 'crime': 'gunah', 'confession': 'iqraar',
         'underground': 'andhere mein', 'poverty': 'garibi', 'soul': 'rooh'}
    ),
    'chekhov': (
        'understated, melancholic, quietly ironic',
        ('Anton Chekhov style: Nothing is stated directly — everything is implied. '
         'Characters talk about trivial things while feeling enormous emotions. '
         'Endings don\'t resolve — they reveal. Short, precise sentences. '
         'Provincial Russian life, quiet desperation, hopes that quietly die. '
         'Hinglish: preserve the SILENCE between the lines. What\'s NOT said matters. '
         'Understate everything. Never explain the emotion — show the gesture.'),
        'literary fiction',
        {'samovar': 'chai ki ketli', 'dacha': 'sheher se door ghar', 'province': 'chota shahar'}
    ),
    'flaubert': (
        'precise, ironic, psychologically acute',
        ('Gustave Flaubert style: Perfect prose — every word earns its place. '
         'Brutal irony about bourgeois aspirations and romantic illusions. '
         'Emma Bovary\'s fantasies vs. the dreary reality of provincial France. '
         'Free indirect discourse — we\'re inside the character\'s deluded thoughts. '
         'Hinglish: keep the ironic distance. The narrator is not sympathetic but not cruel. '
         'Emma\'s romantic daydreams in lush Hinglish, reality in flat Hinglish.'),
        'literary fiction',
        {'bourgeois': 'aam-khate-peete log', 'province': 'chota kasba', 'romance': 'tamanna'}
    ),
    'hugo': (
        'grand, humanist, morally passionate',
        ('Victor Hugo style: Grand historical sweep + intimate human tragedy. '
         'Champions of justice vs. cold institutions. Digressions are part of the rhythm. '
         'Characters are archetypes of human virtues and vices. '
         'Hinglish: maintain the noble register. Jean Valjean speaks with dignity. '
         'The social critique must land. Keep the sweep and the warmth.'),
        'adventure',
        {'inspector': 'inspector sahab', 'convict': 'qaidi', 'bishop': 'padri sahab',
         'barricade': 'barricade', 'revolution': 'inquilab'}
    ),
    'maupassant': (
        'cynical, efficient, socially sharp',
        ('Guy de Maupassant style: Short, brutal, true. No sentimentality. '
         'Human nature laid bare — vanity, class, lust, money. '
         'Endings that hit like a slap. Efficient prose — nothing extra. '
         'Hinglish: keep it lean and a little cold. The irony is in the efficiency.'),
        'literary fiction',
        {}
    ),
    'soseki': (
        'introspective, melancholic, culturally dissonant',
        ('Natsume Soseki style: Meiji-era Japan — West meets East, modern anxiety. '
         'Characters trapped between tradition and modernity. Inner alienation. '
         'Quiet, precise, emotionally restrained. '
         'Hinglish: preserve the restraint. The emotional pain is never shouted.'),
        'literary fiction',
        {'sensei': 'Sensei', 'samurai': 'samurai', 'Meiji': 'Meiji era'}
    ),
}

# Genre vocabulary defaults
GENRE_VOCAB = {
    'mystery/detective': {
        'deduction': 'deduction', 'the evidence': 'saboot', 'clue': 'suraag',
        'inspector': 'inspector sahab', 'the case': 'case', 'motive': 'wajah',
        'alibi': 'alibi', 'the suspect': 'shak ke daayre mein aane wala', 'investigation': 'jaanch',
        'the mystery': 'raaz', 'solution': 'hal',
    },
    'gothic/horror': {
        'the creature': 'woh ajeeb makhluq', 'the monster': 'woh rakshas',
        'the castle': 'woh purana qila', 'darkness': 'ghup andhera',
        'the grave': 'qabr', 'supernatural': 'anokha', 'terror': 'khouf',
        'the curse': 'baddua', 'the shadow': 'parchhaayi', 'the ghost': 'bhoot',
    },
    'romance/social': {
        'the marriage': 'shaadi', 'the proposal': 'rishta ka proposal',
        'society': 'samaj ke log', 'fortune': 'daulat',
        'an eligible man': 'ek achha rishta', 'reputation': 'izzat',
        'the ball': 'dance party', 'the estate': 'haveli aur zameen',
    },
    'adventure': {
        'the quest': 'woh mission', 'the battle': 'jung', 'the sword': 'talwar',
        'the enemy': 'dushman', 'the captain': 'captain sahab',
        'treasure': 'khazana', 'the expedition': 'safar',
    },
    'folk tale / fairy tale': {
        'the king': 'raja sahab', 'the queen': 'rani sahiba',
        'the prince': 'rajkumar', 'the princess': 'rajkumari',
        'the witch': 'daayan', 'the forest': 'jungle', 'the village': 'gaon',
        'the spell': 'jadoo', 'happily ever after': 'phir woh khushi-khushi rehne lage',
    },
    'literary fiction': {}, 'general fiction': {},
}

_STOP = {
    'The','This','That','These','Those','There','Here','When','Where','What',
    'Which','Who','How','Why','And','But','Or','For','As','At','By','In','Of',
    'On','To','It','He','She','We','They','My','Your','Mr','Mrs','Miss','Dr',
    'Sir','Then','Now','Just','Well','Very','Good','Little','Old','New','First',
    'Last','Next','Same','Even','Still','Again','Only','Always','Never','Every',
    'After','Before','With','About','Over','Into','From','Back','Down','Up','Out',
}

def _detect_genre(text):
    t = text.lower()
    if any(w in t for w in ['detective','clue','mystery','suspect','murder','crime','inspector']): return 'mystery/detective'
    if any(w in t for w in ['monster','vampire','ghost','horror','terror','creature','curse']): return 'gothic/horror'
    if any(w in t for w in ['love','proposal','marriage','society','fortune','darling','romance']): return 'romance/social'
    if any(w in t for w in ['king','queen','prince','princess','witch','spell','once upon']): return 'folk tale / fairy tale'
    if any(w in t for w in ['sword','battle','quest','army','expedition','treasure']): return 'adventure'
    return 'literary fiction'

def _extract_chars(text):
    titled = re.findall(
        r'(?:Mr\.?|Mrs\.?|Miss|Dr\.?|Sir|Lord|Lady|Captain|Inspector|'
        r'Professor|Colonel|Major|General|Count|Countess|Father|Mother)\s+'
        r'[A-Z][a-z]{2,}(?:\s+[A-Z][a-z]{2,})?', text)
    singles = re.findall(r'\b[A-Z][a-z]{2,}(?:\s+[A-Z][a-z]{2,})?\b', text)
    counts = Counter(singles + titled)
    return [n for n,c in counts.most_common(20)
            if c >= 2 and n not in _STOP and len(n.split()[-1]) > 2][:8]

def _extract_setting(text):
    m = re.search(
        r'(?:at|in|inside|within|outside|near|entered|through|arrived at|left)\s+'
        r'(?:the\s+)?([A-Z][A-Za-z\s]{2,30}'
        r'(?:Street|Road|House|Hall|Room|Study|Garden|Park|Inn|Hotel|'
        r'Station|Office|Club|Square|Lane|Bridge|Castle|Manor|Lodge|'
        r'Village|Court|Tower|Library|Cottage|Drawing.?Room|Sitting.?Room))',
        text)
    return m.group(1).strip()[:60] if m else ''

def _english_fallback(text):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    for s in sents:
        if len(s.split()) > 6: return s.strip()[:120]
    return ' '.join(text.split()[:25])

_SPEECH_PATTERNS = {
    'formal':     ['I beg your pardon','I must inform','I am afraid','I assure you','It is my duty','permit me'],
    'analytical': ['the evidence','it is clear','I observe','I deduce','therefore','logically','it follows'],
    'anxious':    ['I fear','what if','surely not','I cannot bear','what shall','I dread'],
    'commanding': ['you will','do it at once','I demand','see to it','at once','immediately'],
    'gentle':     ['my dear','pray','I hope','would you','I beg','if you please'],
    'sarcastic':  ['how charming','how delightful','I am sure','how very','how convenient'],
    'lyrical':    ['I wonder','how strange','the light','the silence','the heart'],
}

def _infer_speech_style(char_name, text):
    pattern = re.compile(
        r'["\u201c\u201d][^"\u201c\u201d]{5,200}["\u201c\u201d]'
        r'.*?' + re.escape(char_name.split()[-1]),
        re.IGNORECASE
    )
    dialogue_text = ' '.join(pattern.findall(text)).lower()
    if not dialogue_text: return ''
    scores = {style: sum(1 for kw in kws if kw in dialogue_text)
              for style, kws in _SPEECH_PATTERNS.items()}
    detected = [s for s, sc in scores.items() if sc > 0]
    return ', '.join(detected[:3]) if detected else 'natural'


def update_story_state(english_chunk, hinglish_chunk, chunk_idx,
                       scene_type='DAILY_LIFE', plot_note_en=''):
    global STORY_STATE
    if chunk_idx == 1 and not STORY_STATE['genre']:
        set_genre(_detect_genre(english_chunk))
    for name in _extract_chars(english_chunk):
        if name not in STORY_STATE['characters']:
            STORY_STATE['characters'][name] = {'role':'character','address':'aap','notes':''}
    for name in STORY_STATE['characters']:
        inferred = _infer_speech_style(name, english_chunk)
        if inferred and inferred != 'natural':
            if not STORY_STATE['character_speech_styles'].get(name):
                STORY_STATE['character_speech_styles'][name] = inferred
    s = _extract_setting(english_chunk)
    if s: STORY_STATE['current_setting'] = s
    STORY_STATE['dominant_tone'] = scene_type
    summary = (plot_note_en.strip() if plot_note_en and len(plot_note_en.split()) > 4
               else _english_fallback(english_chunk))
    # v5.0: rolling 3-entry summary
    STORY_STATE['story_so_far'].append({'chunk': chunk_idx, 'summary': summary})
    if len(STORY_STATE['story_so_far']) > 3:
        STORY_STATE['story_so_far'] = STORY_STATE['story_so_far'][-3:]
    STORY_STATE['chunk_count'] = chunk_idx


def build_context_prompt():
    """v5.0 — Compressed context, capped at ~800 chars. Model never overloaded."""
    s = STORY_STATE
    if s['chunk_count'] == 0 and not s['characters'] and not s['established_vocab']:
        return ''
    parts = ['=== STORY CONTEXT (model ke liye sirf — output mein mat daalo) ===']
    info = []
    if s.get('book_title'): info.append(f"Book: {s['book_title']}")
    if s.get('genre'):      info.append(f"Genre: {s['genre']}")
    if s.get('author'):     info.append(f"Author: {s['author']}")
    if info: parts.append(' | '.join(info))
    # Author DNA — single line
    if s.get('author_dna'):
        dna_line = s['author_dna'][:200]
        parts.append(f"AUTHOR STYLE: {dna_line}")
    # Tone anchor
    if s.get('tone_anchor'):
        parts.append(f"TONE ANCHOR (kabhi mat badlo): {s['tone_anchor']}")
    # Characters — compact
    if s['characters']:
        parts.append('CHARACTERS:')
        for name, info in list(s['characters'].items())[:8]:
            style = s['character_speech_styles'].get(name, '')
            line = f"  • {name} ({info['role']}) | {info['address']}"
            if style: line += f" | {style}"
            parts.append(line)
    # Vocab — top 10 only
    if s.get('established_vocab'):
        vocab_items = list(s['established_vocab'].items())[:10]
        parts.append('VOCAB: ' + ' · '.join(f'"{e}"→"{h}"' for e,h in vocab_items))
    # Rolling 3-event summary
    if s['story_so_far']:
        parts.append('RECENT EVENTS:')
        for entry in s['story_so_far']:
            txt = entry['summary'] if isinstance(entry, dict) else str(entry)
            parts.append(f"  C{entry.get('chunk','?') if isinstance(entry,dict) else '?'}: {txt[:80]}")
    parts.append('=== CONTEXT END ===')
    raw = '\n'.join(parts)
    # Hard cap at 800 chars
    return raw[:800] if len(raw) > 800 else raw


def save_state(filepath):
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(STORY_STATE, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f'[!] State save: {e}')


def load_state(filepath):
    global STORY_STATE
    try:
        if os.path.exists(filepath):
            with open(filepath, 'r', encoding='utf-8') as f:
                loaded = json.load(f)
            for key in ['character_speech_styles', 'author_dna', 'tone_anchor', 'author', 'resume_from_chunk']:
                if key not in loaded: loaded[key] = '' if key != 'resume_from_chunk' else 0
            STORY_STATE = loaded
            print(f'[OK] State loaded — chunk {STORY_STATE["chunk_count"]}, '
                  f'{len(STORY_STATE["characters"])} chars, '
                  f'{len(STORY_STATE["character_speech_styles"])} speech styles')
            return True
    except Exception as e:
        print(f'[!] load_state: {e}')
    return False


def _load_latest_state(output_dir='./translation_output'):
    import glob
    files = sorted(glob.glob(f'{output_dir}/state_*.json'))
    if files: return load_state(files[-1])
    return False


def reset_for_new_book(title='', genre='', author=''):
    global STORY_STATE
    STORY_STATE = {
        'book_title': title, 'genre': '', 'author': author,
        'author_dna': '', 'tone_anchor': '',
        'characters': {}, 'character_speech_styles': {},
        'current_setting': '', 'dominant_tone': 'DAILY_LIFE',
        'established_vocab': {}, 'story_so_far': [],
        'chunk_count': 0, 'total_chunks': 0, 'resume_from_chunk': 0,
    }
    if genre: set_genre(genre)
    msg = f'[OK] Reset → "{title}"' if title else '[OK] State reset.'
    print(msg + (f' | Genre: {genre}' if genre else '') + (f' | Author: {author}' if author else ''))


def load_author_dna(author_key):
    """v5.0: Load pre-built AuthorDNA profile."""
    key = author_key.lower().strip()
    if key not in AUTHOR_DNA:
        print(f'[WARN] Author "{key}" not in AuthorDNA. Available: {list(AUTHOR_DNA.keys())}')
        return
    tone_anchor, dna_desc, genre_hint, extra_vocab = AUTHOR_DNA[key]
    STORY_STATE['tone_anchor'] = tone_anchor
    STORY_STATE['author_dna']  = dna_desc
    STORY_STATE['author']      = key.capitalize()
    if not STORY_STATE['genre'] and genre_hint:
        set_genre(genre_hint)
    for eng, hin in extra_vocab.items():
        if eng not in STORY_STATE['established_vocab']:
            STORY_STATE['established_vocab'][eng] = hin
    print(f'[OK] AuthorDNA loaded: {key.upper()}')
    print(f'     tone_anchor: {tone_anchor}')
    print(f'     genre: {STORY_STATE["genre"]}')
    print(f'     vocab: {len(extra_vocab)} entries loaded')


def set_genre(genre):
    STORY_STATE['genre'] = genre
    vocab = GENRE_VOCAB.get(genre, {})
    for eng, hin in vocab.items():
        if eng not in STORY_STATE['established_vocab']:
            STORY_STATE['established_vocab'][eng] = hin
    print(f'[OK] Genre: {genre}' + (f' | {len(vocab)} vocab defaults' if vocab else ''))

def set_book_title(title): STORY_STATE['book_title'] = title; print(f'[OK] Book: "{title}"')

def add_vocab_entry(english, hinglish):
    if english.strip().lower() == hinglish.strip().lower():
        print(f'[SKIP] Same-value: "{english}"'); return
    STORY_STATE['established_vocab'][english] = hinglish
    print(f'[OK] Vocab: "{english}" → "{hinglish}"')

def update_character(name, role=None, address=None, notes=None, speech_style=None):
    if name not in STORY_STATE['characters']:
        STORY_STATE['characters'][name] = {'role':'character','address':'aap','notes':''}
    e = STORY_STATE['characters'][name]
    if role:    e['role']    = role
    if address: e['address'] = address
    if notes:   e['notes']   = notes
    if speech_style: STORY_STATE['character_speech_styles'][name] = speech_style
    print(f'[OK] Char: {name} → {e}' + (f' | speech: {speech_style}' if speech_style else ''))

def print_state_summary():
    s = STORY_STATE
    print('\n=== STORY STATE v5.0 ===')
    if s.get('book_title'):  print(f'  Book    : {s["book_title"]}')
    if s.get('author'):      print(f'  Author  : {s["author"]}')
    if s.get('genre'):       print(f'  Genre   : {s["genre"]}')
    if s.get('tone_anchor'): print(f'  Tone    : {s["tone_anchor"]}')
    print(f'  Chunks  : {s["chunk_count"]} / {s["total_chunks"]}')
    print(f'  Chars   : {len(s["characters"])} | Speech: {len(s["character_speech_styles"])}')
    print(f'  Vocab   : {len(s["established_vocab"])} entries')
    if s['story_so_far']:
        print(f'  Last 3  : ' + ' → '.join(e["summary"][:40] for e in s['story_so_far'] if isinstance(e,dict)))
    print('========================')


print('[OK] Story State Engine v5.0 loaded')
print('  AuthorDNA profiles:', ', '.join(AUTHOR_DNA.keys()))
print('  New: rolling 3-chunk context · context_compression · tone_anchor · resume-from-chunk')

[OK] Story State Engine v5.0 loaded
  AuthorDNA profiles: kafka, austen, doyle, tagore, stoker, tolstoy, dostoevsky, chekhov, flaubert, hugo, maupassant, soseki
  New: rolling 3-chunk context · context_compression · tone_anchor · resume-from-chunk


## 🔩 Step 6 — Load Translation Engine v10.0

**2-Pass Architecture:**
1. **Step 1** — Dead-boring faithful base (temp=0.15, zero style, zero personality)
2. **Step 2** — Hinglish style filter with full ToneGuard Pro + 8 genre few-shots

**New in v10.0:**
- 8 curated Hinglish few-shots (Kafka · Austen · Doyle · Tagore · Stoker · Action · Emotional · Philosophical)
- Anti-hallucination layer: explicit zero-fabrication rule in both passes
- T4 Session Guard: auto-checkpoint, adaptive chunk shrink, timeout warning
- Background I/O: file writes off main thread

In [ ]:
import os, sys, json, time, warnings, re, threading
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')
from IPython.display import display, HTML, clear_output

# ═══════════════════════════════════════════════════════════════════
# JARVIS DISPLAY HELPERS
# ═══════════════════════════════════════════════════════════════════
def _j_bar(pct, w=180, fg='#c0392b', bg='#1a0505'):
    filled = max(0, min(int(w*(pct or 0)/100), w))
    return ('<div style="background:'+bg+';border:1px solid #2a1010;border-radius:3px;'
            'width:'+str(w)+'px;height:12px;display:inline-block;vertical-align:middle;">'
            '<div style="background:linear-gradient(90deg,#7b0000,'+fg+');'
            'width:'+str(filled)+'px;height:100%;border-radius:3px;"></div></div>')

def _j_col(u): return '#555' if u is None else ('#4CAF50' if u<40 else ('#FFD700' if u<75 else '#c0392b'))
def _j_fmt(s):
    s=max(0,int(s)); h,rem=divmod(s,3600); m,sec=divmod(rem,60)
    return (str(h)+'h '+str(m).zfill(2)+'m') if h else (str(m).zfill(2)+'m '+str(sec).zfill(2)+'s')
def _j_spark(vals):
    if not vals: return ''
    blk=' '+''.join(chr(c) for c in [9601,9602,9603,9604,9605,9606,9607,9608])
    mn,mx=min(vals),max(vals); rng=mx-mn or 1
    return ''.join(blk[min(8,int((v-mn)/rng*8))] for v in vals[-60:])
def _j_gpu():
    import subprocess
    try:
        out=subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu','--format=csv,noheader,nounits'],stderr=subprocess.DEVNULL,timeout=3).decode().strip().split(',')
        return float(out[0]),float(out[1])/1024,float(out[2])/1024,float(out[3])
    except: return None,None,None,None

def _jarvis_header(title, subtitle=''):
    display(HTML('<div style="background:linear-gradient(135deg,#0a0a0f,#1a0505);border:2px solid #c0392b;border-radius:8px;padding:14px 20px;font-family:Courier New,monospace;margin:6px 0;">'
                '<div style="color:#c0392b;font-size:1.2em;font-weight:bold;letter-spacing:3px;text-shadow:0 0 10px #c0392b;">'+title+'</div>'
                '<div style="color:#FFD700;font-size:0.8em;margin-top:4px;">'+subtitle+'</div>'
                '</div>'))
def _jarvis_ok(msg):
    display(HTML('<div style="font-family:Courier New,monospace;background:#0a0f0a;border-left:4px solid #4CAF50;padding:5px 14px;margin:2px 0;color:#4CAF50;font-size:0.88em;">[OK] '+str(msg)+'</div>'))
def _jarvis_info(msg, color='#FFD700'):
    display(HTML('<div style="font-family:Courier New,monospace;background:#0a0a0f;border-left:4px solid '+color+';padding:5px 14px;margin:2px 0;color:'+color+';font-size:0.88em;">[·] '+str(msg)+'</div>'))
def _jarvis_warn(msg):
    display(HTML('<div style="font-family:Courier New,monospace;background:#1a0505;border-left:4px solid #e74c3c;padding:5px 14px;margin:2px 0;color:#e74c3c;font-size:0.88em;">[!] '+str(msg)+'</div>'))


def _jarvis_chunk_dashboard(
        i, total, scene, in_words, in_chars, out_chars,
        chunk_t, total_t, eta_s, ctimes, errs,
        gpu_u, vr_u, vr_t, gpu_t,
        model_name, target_lang, tier, qa_passed, qa_summary_text='',
        context_chars=0, pass_label='2-pass',
        session_budget=None, session_elapsed=None):
    pct    = i/total*100 if total else 0
    sc     = '#4CAF50' if pct>=100 else '#FFD700'
    vr_pct = (vr_u/vr_t*100) if (vr_u and vr_t) else 0
    gpu_u_d= str(int(gpu_u))+'%' if gpu_u is not None else 'N/A'
    vr_d   = (str(round(vr_u,1))+'/'+str(round(vr_t,1))+' GB') if vr_u else 'N/A'
    spark  = _j_spark(ctimes)
    t_avg  = str(round(sum(ctimes)/len(ctimes),1))+'s' if ctimes else '--'
    exp_s  = str(round(out_chars/in_chars,2))+'x' if in_chars else 'N/A'
    qa_col = '#4CAF50' if qa_passed else '#FFD700'
    qa_icon= '[OK]' if qa_passed else '[QA]'
    qa_txt = qa_summary_text if qa_summary_text else 'All checks passed'
    err_col= '#e74c3c' if errs else '#4CAF50'
    ctx_kb = f'{context_chars//1024}KB' if context_chars else 'n/a'
    # Session Guard
    sg_html = ''
    if session_budget and session_elapsed:
        rem = session_budget - session_elapsed
        sg_col = '#4CAF50' if rem > 1800 else ('#FFD700' if rem > 600 else '#e74c3c')
        sg_html = f'<div style="color:{sg_col};font-size:0.75em;margin-top:4px;">SESSION: {_j_fmt(rem)} remaining</div>'
    p=[]
    p.append('<div style="background:#070710;border:2px solid #c0392b;border-radius:10px;font-family:Courier New,monospace;overflow:hidden;max-width:940px;">')
    p.append('<div style="background:linear-gradient(90deg,#1a0505,#0a0a1a,#1a0505);border-bottom:2px solid #c0392b;padding:10px 18px;display:flex;justify-content:space-between;align-items:center;">')
    p.append('<span style="color:#c0392b;font-size:1.2em;font-weight:bold;letter-spacing:4px;text-shadow:0 0 12px #c0392b;">J.A.R.V.I.S</span>')
    p.append('<span style="color:#2a2a3a;font-size:0.7em;">'+model_name+' | '+tier+' | '+pass_label+' | v10.0</span>')
    p.append('<span style="color:'+sc+';font-weight:bold;font-size:0.85em;">[ TRANSLATING ]</span>')
    p.append('</div>')
    p.append('<div style="padding:12px 18px;border-bottom:1px solid #1a1a2e;">')
    p.append('<div style="display:flex;justify-content:space-between;margin-bottom:5px;">')
    p.append('<span style="color:#FFD700;font-size:0.88em;">CHUNK <span style="color:#e8e8e8;font-size:1.1em;">'+str(i)+'</span><span style="color:#555;">/'+str(total)+'</span>&nbsp;&nbsp;<span style="color:#c0392b;">'+str(round(pct,1))+'%</span></span>')
    p.append('<span style="color:#555;font-size:0.78em;">scene: <span style="color:#e8e8e8;">'+scene+'</span> | '+str(in_words)+'w | exp: <span style="color:#FFD700;">'+exp_s+'</span> | ctx: <span style="color:#4CAF50;">'+ctx_kb+'</span></span>')
    p.append('</div>')
    p.append('<div style="background:#12010a;border:1px solid #2a0a0a;border-radius:4px;padding:2px;">')
    p.append('<div style="background:linear-gradient(90deg,#5a0000,#a00010,#c0392b);height:18px;border-radius:3px;min-width:3px;box-shadow:0 0 10px #c0392b55;width:'+str(round(pct,2))+'%;"></div>')
    p.append('</div>'+sg_html+'</div>')
    p.append('<div style="display:grid;grid-template-columns:repeat(5,1fr);border-bottom:1px solid #1a1a2e;">')
    for label,val in [('ELAPSED',_j_fmt(total_t)),('ETA',_j_fmt(eta_s)),
                       ('LAST',str(round(chunk_t,1))+'s'),('AVG',t_avg),
                       ('ERRORS','<span style="color:'+err_col+';">'+str(errs)+'</span>')]:
        p.append('<div style="background:#0d0d1a;padding:10px 12px;border-right:1px solid #1a1a2e;"><div style="color:#2a2a4a;font-size:0.63em;letter-spacing:2px;">'+label+'</div><div style="color:#FFD700;font-size:1em;font-weight:bold;">'+val+'</div></div>')
    p.append('</div>')
    p.append('<div style="display:grid;grid-template-columns:1fr 1fr;border-bottom:1px solid #1a1a2e;">')
    p.append('<div style="background:#0a0a18;padding:10px 14px;border-right:1px solid #1a1a2e;">'
             '<div style="color:#FFD700;font-size:0.63em;letter-spacing:2px;margin-bottom:6px;">GPU — TESLA T4</div>'
             '<div style="display:flex;align-items:center;gap:6px;margin-bottom:4px;">'
             '<span style="color:#2a2a4a;font-size:0.7em;width:52px;">UTIL</span>'
             +_j_bar(gpu_u,140,'#c0392b')+'<span style="color:'+_j_col(gpu_u)+';font-size:0.78em;">'+gpu_u_d+'</span></div>'
             '<div style="display:flex;align-items:center;gap:6px;">'
             '<span style="color:#2a2a4a;font-size:0.7em;width:52px;">VRAM</span>'
             +_j_bar(vr_pct,140,'#8B0000')+'<span style="color:#e8e8e8;font-size:0.78em;">'+vr_d+'</span></div>'
             '</div>')
    p.append('<div style="background:#0a0a18;padding:10px 14px;">'
             '<div style="color:#FFD700;font-size:0.63em;letter-spacing:2px;margin-bottom:6px;">STATE v5.0 | AuthorDNA | ToneGuard Pro</div>'
             '<div style="color:'+qa_col+';font-size:0.8em;">'+qa_icon+' '+qa_txt+'</div>'
             '<div style="color:#2a2a4a;font-size:0.63em;margin-top:6px;">TIMING SPARK</div>'
             '<div style="color:#8B0000;font-family:monospace;font-size:0.88em;">'+(spark or '--')+'</div>'
             '<div style="color:#2a2a4a;font-size:0.63em;">avg '+t_avg+'</div>'
             '</div>')
    p.append('</div>')
    p.append('<div style="background:#050508;padding:4px 18px;"><span style="color:#18080a;font-size:0.63em;">STARK INDUSTRIES v10.0 | AuthorDNA | 8-GenreFewShots | AntiHallucination | T4 SessionGuard</span></div></div>')
    display(HTML(''.join(p)))


def _jarvis_complete(model_name, total_time, n_chunks, orig_chars, trans_chars,
                     deva_count, qa_failed, output_file, state_file=''):
    ratio = round(trans_chars/orig_chars,2) if orig_chars else 0
    qa_col= '#FFD700' if qa_failed else '#4CAF50'
    qa_txt= ('WARN '+str(qa_failed)+' chunks') if qa_failed else 'PASS — All clear'
    dv_col= '#e74c3c' if deva_count else '#4CAF50'
    dv_txt= ('WARN '+str(deva_count)+' chunks') if deva_count else 'PASS — Clean'
    out_nm= str(output_file).split('/')[-1]
    p=[]
    p.append('<div style="background:#070710;border:2px solid #4CAF50;border-radius:10px;font-family:Courier New,monospace;overflow:hidden;max-width:940px;margin-top:10px;">')
    p.append('<div style="background:linear-gradient(90deg,#0a1a0a,#0a0a1a,#0a1a0a);border-bottom:2px solid #4CAF50;padding:12px 20px;">'
             '<span style="color:#4CAF50;font-size:1.25em;font-weight:bold;letter-spacing:3px;text-shadow:0 0 12px #4CAF50;">[ MISSION ACCOMPLISHED ]</span></div>')
    p.append('<div style="display:grid;grid-template-columns:1fr 1fr;padding:16px 20px;gap:16px;">')
    p.append('<table style="color:#e8e8e8;font-size:0.88em;border-collapse:collapse;">')
    rows=[('Total Time',_j_fmt(total_time)),('Chunks',str(n_chunks)),
          ('Avg/Chunk',_j_fmt(total_time/n_chunks) if n_chunks else '--'),
          ('Input',format(orig_chars,',')+' chars'),('Output',format(trans_chars,',')+' chars'),
          ('Expansion',str(ratio)+'x')]
    for k,v in rows: p.append(f'<tr><td style="color:#FFD700;padding:3px 14px 3px 0;">{k}</td><td>{v}</td></tr>')
    p.append('</table><table style="color:#e8e8e8;font-size:0.88em;border-collapse:collapse;">')
    for k,c,v in [('Devanagari',dv_col,dv_txt),('QA Pipeline',qa_col,qa_txt)]:
        p.append(f'<tr><td style="color:#FFD700;padding:3px 14px 3px 0;">{k}</td><td style="color:{c};">{v}</td></tr>')
    p.append(f'<tr><td style="color:#FFD700;padding:3px 14px 3px 0;">File</td><td style="color:#888;font-size:0.85em;">{out_nm}</td></tr>')
    p.append('</table></div>')
    p.append('<div style="background:#050a05;border-top:1px solid #1a3a1a;padding:5px 20px;"><span style="color:#1a3a1a;font-size:0.65em;">STARK INDUSTRIES v10.0 | AuthorDNA | ToneGuard Pro | Anti-Hallucination</span></div></div>')
    display(HTML(''.join(p)))


import torch
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
_gpu_nm  = torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'N/A'
_gpu_mem = f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB" if DEVICE=='cuda' else 'N/A'
_jarvis_header('STARK INDUSTRIES — NEURAL TRANSLATION ARRAY v10.0',
               f'Device: {DEVICE.upper()} · {_gpu_nm} · {_gpu_mem} | Model: gemma3:27b | 2-Pass | AuthorDNA | ToneGuard Pro')


# ═══════════════════════════════════════════════════════════════════
# SOURCE CLEANER
# ═══════════════════════════════════════════════════════════════════
_GERMAN_RESIDUALS = {
    r'\bMutter\b':'Maa', r'\bVater\b':'Papa', r'\bHerr\b':'Mr.',
    r'\bFrau\b':'Mrs.', r'\bFräulein\b':'Miss', r'\bGott\b':'Bhagwan',
    r'\bJa\b':'Haan', r'\bNein\b':'Nahi', r'\bDanke\b':'Shukriya',
    r'\bBitte\b':'Please', r'\bAch\b':'Ugh', r'\bProcurator\b':'Chief Clerk',
    r'\bProkurist\b':'Chief Clerk', r'Mr\. Procurator':'Mr. Chief Clerk',
}
def _strip_german_residuals(text):
    for p, r in _GERMAN_RESIDUALS.items():
        text = re.sub(p, r, text, flags=re.IGNORECASE)
    return text

def _preprocess_english_source(text):
    text = re.sub(r'^TRANSLATED TO ENGLISH\b.*?(?:={10,}|-{10,})\n+','', text, flags=re.DOTALL)
    text = _strip_german_residuals(text)
    text = re.sub(r'[\u0900-\u097F]+', '', text)
    return text.strip()

def clean_source_text(text):
    text = re.sub(r'(?m)^[A-Z][A-Za-z \'\,\-]{3,}\s+by\s+[A-Z][A-Za-z .]{2,}\s+\d{1,4}\s*$', '', text)
    text = re.sub(r'(?m)^\s*Page\s+\d{1,4}\s*$', '', text)
    text = re.sub(r'(?m)^\s*\d{1,4}\s*$', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


# ═══════════════════════════════════════════════════════════════════
# SCENE DETECTOR
# ═══════════════════════════════════════════════════════════════════
_SCENE_KW = {
    'DEDUCTION':  ['deduce','observe','perceive','infer','conclude','obvious','clue','evidence','reasoning','logically','therefore','solve','mystery'],
    'REVELATION': ['suddenly realised','suddenly realized','all became clear','it dawned','the truth','secret','revealed','discovered','confession','admitted','at last'],
    'TENSION':    ['crept','tiptoed','lurked','peered','silence fell','a shadow','locked','dare not','warning','danger','trap','ambush','suspicion','uneasy','dread'],
    'ACTION':     ['charged','attacked','fired','shot','chased','fled','fight','struck','wounded','escaped','panic','crashed','rushed','dashed','sprang','leaped','flung','seized'],
    'EMOTIONAL':  ['wept','cried','sobbed','tears','grief','sorrow','mourning','heartbroken','tragedy','died','death','loss','loved','happiness','joy','reunion','farewell','forgive'],
    'BANTER':     ['laughed','chuckled','smiled','grinned','teased','merry','amusing','absurd','funny','joke','wit','clever','nonsense','irony','sarcasm','playful'],
    'CONFRONTATION':['how dare','you lied','you deceived','confess','deny','accused','confronted','guilty','cornered','betrayed','traitor','fury','rage','demanded','shouted'],
    'HORROR':     ['monster','creature','beast','demon','ghost','horror','terror','nightmare','blood','corpse','grave','tomb','darkness','supernatural','curse','haunted'],
    'ROMANCE':    ['love','heart','desire','passion','longing','beautiful','blush','admire','enchanted','proposal','beloved','darling','dearest','kiss','embrace'],
    'PHILOSOPHICAL':['meaning','purpose','existence','fate','destiny','justice','morality','conscience','soul','freedom','truth','wisdom','contemplate','ponder','reflect'],
    'DESCRIPTION':['the room','the house','the garden','the street','morning','evening','sunset','dawn','rain','storm','wind','snow','fog','mountain','river','sea','forest'],
    'INNER_MONOLOGUE':['I thought','I felt','I wondered','I knew','I could not','it seemed to me','I asked myself','as if','something inside','part of me'],
    'SPEECH':     ['my friends','let me explain','i shall','we must','i assure you','address','declare','proclaim','announce'],
}
def detect_scene_type(text):
    tl = text.lower()
    scores = {sc: sum(2 if ' ' in kw else 1 for kw in kws if kw in tl) for sc,kws in _SCENE_KW.items()}
    best = max(scores, key=scores.get)
    return 'DAILY_LIFE' if scores[best]==0 else best

SCENE_CTX = {
    'DEDUCTION':     'DEDUCTION — Precise, confident. Clue by clue. Methodical.',
    'REVELATION':    'REVELATION — Short lines. Dramatic pause. Reader shocked.',
    'TENSION':       'TENSION — Slow. Cold. Short sentences. Dread builds.',
    'ACTION':        'ACTION — Fast. Punchy. 4-6 words per sentence.',
    'EMOTIONAL':     'EMOTIONAL — Slow, heavy. Let it breathe. Silences matter.',
    'BANTER':        'BANTER — Light, witty. Mazaa aaye.',
    'CONFRONTATION': 'CONFRONTATION — Direct, sharp. High stakes.',
    'SPEECH':        'SPEECH — Authority. Weight. Build-up.',
    'HORROR':        'HORROR — Dark, visceral, creepy. Reader uncomfortable.',
    'ROMANCE':       'ROMANCE — Warm, intimate. Feelings unfold slowly.',
    'PHILOSOPHICAL': 'PHILOSOPHICAL — Deep but accessible. Simple words, heavy meaning.',
    'DESCRIPTION':   'DESCRIPTION — Vivid, sensory. Take reader there.',
    'INNER_MONOLOGUE':'INNER MONOLOGUE — Fragmented, honest, contradictory.',
    'DAILY_LIFE':    'DAILY LIFE — Conversational, easy flow.',
}


# ═══════════════════════════════════════════════════════════════════
# THE HINGLISH VOICE GUIDE (researched)
# ─────────────────────────────────────────────────────────────────
# Based on: OTT India (Made in Heaven, Little Things, Panchayat),
#           educated urban Indian 25-35 WhatsApp register,
#           Hinglish literary tradition (Shobha De onward),
#           modern reading preferences (audiobook/ebook compatible).
#
# The TARGET voice: how an educated Indian millennial (Mumbai/Delhi/
# Bangalore, 28 years old) would narrate a story to a friend —
# smart, clear, emotionally present, never trying too hard.
# ═══════════════════════════════════════════════════════════════════

_HINGLISH_VOICE_GUIDE = """
=== HINGLISH VOICE — MILLENNIAL INDIA (v10.0 RESEARCHED) ===

TARGET REGISTER:
Like an educated Indian in their late 20s / early 30s narrating a story to a friend.
Think: Made in Heaven, Little Things, Scam 1992 voiceover — clear, warm, intelligent.
NOT: a stand-up comedian, a WhatsApp forward, or a college kid.

WHAT THIS SOUNDS LIKE:
• Mostly Hindi/Urdu vocabulary, English nouns/concepts where natural
• Sentences flow like speech — not textbook Hindi, not pidgin English
• Emotions felt fully — but expressed with restraint, not melodrama
• Formal characters speak with dignity — not casually addressed
• Simple words chosen over complex ones — but precision kept

APPROVED CONNECTORS (use sparingly, where natural):
matlab · seedha · basically · obviously · waise · phir bhi · lekin
aur · isliye · kyunki · aakhirkar · tab · tab jaake · phir

APPROVED FILLER PHRASES (only where organic):
ek pal ke liye · dheere dheere · aahista · achanak · seedha jawab tha

WHAT TO NEVER USE:
✗ yaar (addressing anyone) · bhai (addressing anyone)
✗ tu / tujhe / tera / tune (always tum/tumhe/tumhara/tumne)
✗ gadha / bewakoof / ullu (abusive/dismissive tone)
✗ 'like totally' · 'no cap' · English Gen-Z slang
✗ forced rhymes or Bollywood-isms
✗ exclamation-heavy language (!!! = wrong tone)
✗ adding narrator commentary not in the source

ADDRESS FORMS:
• Formal / respectful → aap / aapko / aapka
• Familiar / equal → tum / tumhe / tumhara / tumne
• NEVER tu unless character explicitly uses it AND source shows it
"""


# ═══════════════════════════════════════════════════════════════════
# CURATED FEW-SHOT EXAMPLES — 8 genre/author styles
# Each example: BASE (Step1 output) → HINGLISH (Step2 target)
# These show EXACTLY the right voice for each genre.
# ═══════════════════════════════════════════════════════════════════

_FEW_SHOTS = """
=== GENRE FEW-SHOTS — EXACT VOICE FOR EACH SCENE TYPE ===
(Yeh examples model ke liye hain — output mein mat daalo)

── EXAMPLE 1: LITERARY ABSURDISM (Kafka style) ──
BASE:
Ek subah Gregor Samsa jab sapne se utha, usne dekha ki woh ek bade keede mein badal gaya tha.
Woh apni sakht, zirah jaisi peeth ke bal leta hua tha, aur sar thoda uthaya toh usne apna bhoora,
gumbad sa pet dekha jiske upar razai tiki thi aur girne hi wali thi.
RIGHT HINGLISH:
Ek subah Gregor Samsa uthaa — aur usne paya ki woh ek bade keede mein badal gaya tha.
Peeth sakht thi, jaise koi zirah pohe ho. Sir uthaya toh saamne tha woh bhoora, gumbad-sa pet,
jiski takediaar dhaarivali aakaar thi. Upar razai tiki thi — girti girti bach rahi thi.
WRONG (do NOT write like this):
"Yaar, ek din Gregor Samsa uthaa aur usne socha: what the hell! Main toh ek keeda ban gaya hoon!"
WHY WRONG: Tone added that source doesn't have. Kafka is flat, emotionless. Horror = flatness.

── EXAMPLE 2: SOCIAL COMEDY (Austen style) ──
BASE:
Yeh toh sabko pata hai ki ek ameer aur kunwara mard ko ek biwi ki zaroorat hoti hai.
Is baat ko is qadar maan liya gaya tha ki aise kisi bhi mard ko seedha iss nazar se dekha jata tha
ki woh kaisi aurat chahta hai.
RIGHT HINGLISH:
Yeh baat toh samaj mein sab jaante hain — ek daulat wala kunwara mard ko biwi ki zaroorat hoti hi hai.
Is sach ko itna pakka maan liya gaya tha ki jab bhi aisa koi mard aas-paas aata, sabki nazar seedha
uski taraf hoti — bhale hi usne khud kuch bola na ho.
WRONG:
"Bhai sach baat yeh hai: agar koi rich bachelor hai toh woh clearly ek wife dhundh raha hai. Obviously."
WHY WRONG: Austen's irony is gentle, not loud. The humor is in understatement, not in announcing it.

── EXAMPLE 3: DETECTIVE/ANALYTICAL (Doyle/Holmes style) ──
BASE:
Main ek aise mard ke saamne tha jo apni ankhon mein koi shak nahi rakhta tha.
Usne ek lambi ungali mere baayin hath ki taraf uthai aur seedha kaha ki woh wahan ka imprinting
dekh sakta hai.
RIGHT HINGLISH:
Woh mard bilkul nirbhay tha — aankhon mein ek bhi shak ka kona nahi.
Usne apni lambi ungali meri baayin taraf uthai aur spashtata se kaha ki woh wahan ki chhap dekh
sakta hai.
WRONG:
"Yaar, woh banda mast confident tha! Usne bola — 'Bhai main dekh sakta hoon ki wahan kya hua!'"
WHY WRONG: Holmes register is formal and precise. "Yaar" and "bhai" completely break the character.

── EXAMPLE 4: LYRICAL/PHILOSOPHICAL (Tagore style) ──
BASE:
Jahan mann mein dar nahi ho aur sar unch uthaya gaya ho.
Jahan gyaan mukt ho. Jahan duniya ko tukdon mein baant kar nahi gaya ho.
RIGHT HINGLISH:
Jahan mann mein koi dar na ho — aur sar uthake jeena ho.
Jahan gyan par koi taala na ho. Jahan is duniya ko chhote-chhote daayron mein na baant diya gaya ho.
WRONG:
"Basically yeh poem mein Tagore bol rahe hain: mind fearless hona chahiye, like, totally free."
WHY WRONG: Destroys the lyricism. Tagore's poetry-prose needs breathing room, not commentary.

── EXAMPLE 5: GOTHIC/HORROR (Stoker style, diary entry) ──
BASE:
Uske kambakht ghere aaya aur mujhe mahsoos hua ki woh meri taraf dekh raha hai.
Mujhe thanda lagne laga, halanki kamra poori tarah band tha.
Ek aisi cheez meri taraf aa rahi thi jo mujhe pehchanti thi — ya shayad pehchanna chahti thi.
RIGHT HINGLISH:
Uske saaye ne mujhe ghera — aur mujhe laga jaise uski nazar mujh par hai.
Thandi mahsoos hone lagi, halanki kamre ki khidkiyan band theen.
Kuch tha jo mere paas aa raha tha — jo mujhe pehchanta tha, ya pehchanna chahta tha.
WRONG:
"OMG yaar woh Count bilkul creepy tha! Mujhe toh darr lag gaya seriously!"
WHY WRONG: Gothic dread builds through description, not exclamation. Coldness IS the horror.

── EXAMPLE 6: ACTION SCENE ──
BASE:
Woh bhaag pada. Seedha paidal path ke baad mein ghusa aur ek deewar se takraya.
Khoon tha. Woh utha aur phir bhaag pada.
RIGHT HINGLISH:
Woh bhaaga. Paidal raaste mein ghusa — ek deewar se takraya.
Khoon tha. Uthaa. Phir bhaaga.
WRONG:
"Woh like totally sprint kar gaya, basically wall se hit hua, blood tha obviously, par woh chalta raha!"
WHY WRONG: Action scenes need SHORT sentences. English fillers kill the pace.

── EXAMPLE 7: EMOTIONAL / TRAGEDY ──
BASE:
Woh nahi rahi. Ghar khali tha. Uski kursi vahan thi, lekin woh nahi thi.
Maine baar baar uski kursi ko dekha, jaise woh aa sakti thi.
RIGHT HINGLISH:
Woh chali gayi thi. Ghar khali tha — seedha, bina kisi aahat ke.
Uski kursi wahin rakhi thi. Maine baar baar wahan dekha... jaise shayad woh aa jaaye.
WRONG:
"Yaar woh toh gone ho gayi. Matlab kitna sad tha na, bohot bura laga uske jaane ke baad!"
WHY WRONG: Grief speaks in objects and silence. Commentary destroys it.

── EXAMPLE 8: INNER MONOLOGUE (Dostoevsky/psychological style) ──
BASE:
Main nahi jaanta tha kya karna chahiye. Main sochta raha, phir ruk gaya. Main galat tha.
Nahi, main galat nahi tha. Lekin phir kya tha? Mujhe kuch nahi samajh aa raha tha.
RIGHT HINGLISH:
Main nahi jaanta tha. Sochta raha — phir ruk gaya.
Main galat tha. Nahi... galat nahi tha. Toh phir kya tha?
Mujhe samajh nahi aa raha tha. Bilkul bhi nahi.
WRONG:
"Basically woh confused tha, uske andar basically ek internal conflict chal raha tha, like obviously!"
WHY WRONG: Inner monologue must feel like real fragmented thought. Summarizing it kills it.

=== FEW-SHOTS END — Ab neeche diye base translation ko in examples jaisi Hinglish mein style karo ===
"""


# ═══════════════════════════════════════════════════════════════════
# STEP 1 SYSTEM — DEAD BORING, FAITHFUL BASE
# Anti-Hallucination rules strengthened in v10.0
# ═══════════════════════════════════════════════════════════════════
STEP1_SYSTEM = """
TASK:
Convert the text into natural Hinglish for audiobook narration.

CORE GOAL:
Preserve intent and context, NOT literal wording.

LANGUAGE RULES:
- Use modern Indian Hinglish (spoken style)
- Hindi sentence structure + natural English words
- Use commonly spoken English (decision, plan, stressed)
- Use Hindi for natural phrases (e.g., "kuch toh off hai")

AVOID:
- Literal translation
- Formal/literary Hindi
- Rare or unnatural English phrases

STYLE RULES:
- Use spoken connectors: "ko leke", "aisa lag raha tha"
- Keep tone conversational but respectful

STRUCTURE RULES:
- Break long sentences into smaller spoken sentences
- Each sentence should be easy to speak in one breath
- Maintain smooth flow

OUTPUT:
Only Hinglish text.
"""
# """You are a mechanical English-to-Hindi translator. Your ONLY job is exact meaning transfer.

# OUTPUT: ONLY the translated text. No labels, no preamble, no commentary.

# === STRICT RULES ===
# ✓ Translate EVERY sentence completely — not one line missed
# ✓ Roman script ONLY — absolutely zero Devanagari characters
# ✓ Preserve the EXACT meaning of every sentence — word for word intent
# ✓ Preserve the EXACT emotional register — formal stays formal, cold stays cold, serious stays serious
# ✓ Preserve dialogue structure exactly — who says what to whom
# ✓ Keep sentence structure close to the original
# ✓ Paragraph breaks: follow the source exactly
# ✓ Gender: male aaya/tha/gaya | female aayi/thi/gayi

# === ANTI-HALLUCINATION HARD RULES ===
# ✗ Do NOT add any sentence, phrase, or word not present in the source
# ✗ Do NOT omit any sentence from the source — every line must appear in output
# ✗ Do NOT invent character actions, emotions, or reactions
# ✗ Do NOT add tone, style, personality, or commentary
# ✗ Do NOT simplify or soften emotions — cold stays cold
# ✗ Do NOT use GenZ words or dramatic embellishment

# Boring + mechanical + complete = CORRECT output for this stage.
# First word of output = first translated word. Nothing before it."""

STEP1_USER = """Translate this English text into simple, neutral Hindi (Roman script only).
Your ONLY job is exact meaning transfer — nothing else.

ANTI-HALLUCINATION RULE: Translate ONLY what is written. Do NOT add or omit anything.
Boring, mechanical, complete output = correct output for this stage.

CONSISTENCY RULES:

- Maintain consistent tone across the entire passage
- Do not switch between formal and casual styles randomly
- Keep character voice consistent (if dialogue present)
- Avoid sudden changes in language style

- Ensure output is clean and properly formatted
- No broken or incomplete sentences

CONTEXT AWARENESS:

- Maintain continuity with previous sentences
- Do not translate each sentence independently
- Keep narrative flow intact

---
{chunk}
---

Hindi translation (Roman script, first word = first translated word):"""


# ═══════════════════════════════════════════════════════════════════
# STEP 2 SYSTEM — HINGLISH STYLE FILTER (ToneGuard Pro)
# v10.0: Hinglish Voice Guide + 8 Few-shots + Anti-Hallucination
# ═══════════════════════════════════════════════════════════════════
STEP2_SYSTEM = """
TASK:
Refine the Hinglish text for audiobook narration.

DO NOT:
- Change meaning
- Re-translate content

IMPROVE:
- Sentence flow (smooth, natural)
- Tone consistency
- Sentence breaking for listening comfort
- Remove awkward phrasing

ENSURE:
- Natural spoken Hinglish
- Easy to understand and listen

CONTEXT AWARENESS:

- Maintain continuity with previous sentences
- Do not translate each sentence independently
- Keep narrative flow intact

CONSISTENCY RULES:

- Maintain consistent tone across the entire passage
- Do not switch between formal and casual styles randomly
- Keep character voice consistent (if dialogue present)
- Avoid sudden changes in language style

- Ensure output is clean and properly formatted
- No broken or incomplete sentences

OUTPUT:
Only improved Hinglish text.
"""

# """Tum ek expert Hinglish style editor ho. Tumhare paas ek clean Hindi base translation hai.
# Tumhara kaam: isko natural, readable Hinglish mein convert karna — BINA emotional register badle.

# {hinglish_voice_guide}

# === ANTI-HALLUCINATION — SABSE ZAROORI RULE ===

# ZERO FABRICATION RULE: Tum sirf jo base translation mein likha hai, wahi Hinglish mein style karo.
# ✗ Koi nayi line mat add karo — source mein jo nahi hai, output mein bhi nahi hoga
# ✗ Koi bhi sentence skip mat karo — har ek line translate honi chahiye
# ✗ Characters ke reactions, emotions ya actions invent mat karo
# ✗ Apni taraf se koi commentary, opinion ya reaction mat daalo
# ✓ SIRF style karo — fabricate mat karo

# === STRICT FIDELITY RULE ===

# Tum kisi bhi sentence ka emotional tone CHANGE NAHI KAR SAKTE.
# Formal → formal hi rahe | Serious → serious hi rahe | Cold → cold hi rahe
# Angry → angry hi rahe | Sad → sad hi rahe | Absurd → absurd hi rahe

# GALAT: 'Please leave the room' → 'Bhai please nikal jao' (tone change — FAIL)
# SAHI:  'Please leave the room' → 'Aap please room se bahar chale jaaiye' ✓

# {tone_rules_section}

# === CHARACTER CONSISTENCY ===

# {char_styles}

# === CONTEXT CONTINUITY ===

# Previous chunk ka tone is chunk mein bhi CONTINUE hoga.
# ✗ Previous chunk serious tha → iss chunk casual NAHI hoga
# ✓ Tone shift SIRF tab karo jab source text mein clearly indicate ho

# === TONE GUARD — PROHIBITED ===

# ✗ Sarcasm add mat karo — unless source itself is sarcastic
# ✗ Attitude ya aggression mat daalo — unless source shows anger
# ✗ Street slang ya GenZ filler mat daalo — unless character speaks that way
# ✗ Irony ya humor add mat karo — unless source is ironic
# ✗ Emotional commentary mat daalo — 'kitna bura hua' type lines = FAIL
# ✓ Style filter = readability improvement ONLY, NOT creativity

# === OUTPUT RULES ===

# R1 — ROMAN SCRIPT ONLY. Ek bhi Devanagari = FAIL.
# R2 — HAR LINE TRANSLATE KARO. Ek bhi line skip = FAIL.
# R3 — PARAGRAPH BREAKS: Har 2-3 sentences ke baad ek blank line.
# R4 — ENGLISH SIRF: proper nouns · naturalized words (police, train, office) · technical terms
# R5 — ADDRESS: tu/tujhe/teri/tera/tune KABHI NAHI. Sirf tum/tumhe/tumhari/tumhara/tumne.
# R6 — GENDER: Male: aaya/tha/gaya | Female: aayi/thi/gayi
# R7 — PASSIVE BANNED: 'mujhe le jaya gaya' → 'woh mujhe le gaye'

# OUTPUT MEIN SIRF TRANSLATED HINGLISH TEXT.
# Pehla word = pehla translated word. Koi prefix nahi. Koi label nahi.


# Convert the following text into natural Hinglish for audiobook narration.

# STRICT INSTRUCTIONS:

# 1. Meaning:
# - Preserve the original intent and context, not the exact wording or structure
# - Do NOT add or remove information

# 2. Language Style:
# - Write in modern Indian Hinglish (spoken, conversational tone)
# - Sentence structure should be Hindi-based
# - Actively replace formal Hindi words with commonly used English words
#   (e.g., decision, plan, stressed, idea, problem)

# 3. Word Selection Rules:
# - Use English for commonly spoken words in India
# - Use Hindi for natural phrases and expressions
# - Avoid literal translations of English phrases
#   (e.g., avoid "something is wrong", prefer "kuch toh off hai")

# 4. Spoken Tone:
# - Use natural spoken connectors:
#   (e.g., "ko leke", "aisa lag raha tha", "samajh nahi aa raha tha")
# - Avoid literary or formal Hindi

# 5. Audiobook Structure (VERY IMPORTANT):
# - Break long sentences into smaller spoken sentences
# - Each sentence should be easy to speak in one breath
# - Prefer medium-length sentences (8–16 words)
# - Maintain smooth flow between sentences

# 6. Tone Control:
# - Slightly modernize the tone for readability
# - Keep respect in language (no "tu", no abuse)
# - Maintain author's intent and emotional depth

# 7. Output Rules:
# - No explanations
# - No extra text
# - Only final Hinglish output

# Goal:
# The output should feel like a modern Indian narrator explaining the same text naturally in Hinglish, suitable for audiobooks.

# Poori translation ke BILKUL AAKHIR mein sirf yeh ek line daalo:
# ##PLOT_NOTE: [1 English sentence: main event of this chunk]"""

STEP2_USER = """{context_block}

SCENE TONE (reference only):
{scene_context}

{few_shots}

=== CHARACTER SPEECH STYLES — STRICTLY ENFORCE ===
(In styles se bahar jaana = FIDELITY VIOLATION)
{char_styles}

=== PREVIOUS CHUNK END — TONE REFERENCE ===
(Is tone ko is chunk mein bhi CONTINUE karo):
{overlap_section}

NEECHE DIYA BASE TRANSLATION KO HINGLISH MEIN STYLE KAR — EK LINE BHI MAT CHHODNA:
---
{base_translation}
---

Hinglish styled output (pehla word = pehla word, koi extra text nahi):"""


# Legacy single-pass prompts
TRANSLATION_PROMPTS = {
    'BASIC': {
        'system': 'You are a professional English-to-Hindi translator.\nOUTPUT: Only the Hinglish translation. RULES: Translate ALL text · Simple words · Roman script · No Devanagari',
        'user': 'Translate to simple Hinglish. Roman script only.\n\n{context_block}\n\nEnglish:\n---\n{chunk}\n---\n\nHinglish:'
    },
    'INTERMEDIATE': {
        'system': 'You are an expert English-to-Hinglish translator.\n\nOUTPUT: ONLY the translated Hinglish text.\n\nRULES:\n✓ Translate every sentence\n✓ Natural conversational flow\n✓ Roman script only — no Devanagari\n✓ Paragraph breaks every 2-3 sentences\n✓ Gender: male aaya/tha | female aayi/thi\n✓ Always tum/tumhe, never tu/tujhe\n✗ No yaar/bhai as address terms\n✗ No Gen-Z slang',
        'user': 'Translate to modern Hinglish. Roman script only.\n\n{context_block}\nSCENE: {scene_context}\nPREVIOUS END: {overlap_section}\n\nEnglish:\n---\n{chunk}\n---\n\nHinglish (ONLY translated text):'
    },
}

LANG_NAMES = {
    'hinglish':'Hinglish','hin_Deva':'Hindi','ben_Beng':'Bengali',
    'tam_Taml':'Tamil','tel_Telu':'Telugu','mar_Deva':'Marathi','guj_Gujr':'Gujarati',
}


# ═══════════════════════════════════════════════════════════════════
# CHUNKING
# ═══════════════════════════════════════════════════════════════════
def chunk_text(text, chunk_words=350):
    paragraphs = re.split(r'\n\s*\n|\r\n\s*\r\n', text)
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    chunks, current_chunk, current_count = [], [], 0
    for para in paragraphs:
        pw = para.split(); pc = len(pw)
        if pc > chunk_words:
            if current_chunk:
                chunks.append('\n\n'.join(current_chunk))
                current_chunk, current_count = [], 0
            for idx in range(0, len(pw), chunk_words):
                chunks.append(' '.join(pw[idx:idx+chunk_words]))
        elif current_count + pc > chunk_words and current_chunk:
            chunks.append('\n\n'.join(current_chunk))
            current_chunk, current_count = [para], pc
        else:
            current_chunk.append(para); current_count += pc
    if current_chunk: chunks.append('\n\n'.join(current_chunk))
    return chunks


# ═══════════════════════════════════════════════════════════════════
# OVERLAP
# ═══════════════════════════════════════════════════════════════════
def get_overlap(prev_translation, overlap_words=80):
    if not prev_translation or overlap_words == 0: return ''
    words = prev_translation.split()
    return ' '.join(words[-overlap_words:]) if len(words) > overlap_words else prev_translation

def build_overlap_section(prev_tail):
    if not prev_tail: return '(Pehla chunk — koi pichla context nahi)'
    return '[PICHLI TRANSLATION KA AAKHRI HISSA — DOBARA TRANSLATE MAT KARO]:\n' + prev_tail + '\n[YAHAN SE NAYI TRANSLATION SHURU KARO]'


# ═══════════════════════════════════════════════════════════════════
# POST-PROCESSING
# ═══════════════════════════════════════════════════════════════════
_H_ALLOW = {
    'nahi','kahi','bhai','yaar','karo','raha','wala','haan','abhi','acha','accha',
    'theek','sahi','dekho','suno','chalo','jao','aao','bolo','batao','kuch','kisi',
    'koi','sab','sabse','bahut','zyada','woh','yeh','uska','uski','mera','meri',
    'tumhara','tumhari','toh','bhi','tha','thi','the','kar','kiya','gaya','gayi',
    'gaye','aaya','aayi','aaye','pehle','baad','saath','andar','bahar','bilkul',
    'ekdum','pakka','lekin','magar','kyunki','isliye','jab','tab','agar','phir',
    'kabhi','poora','poori','thoda','thodi','matlab','basically','obviously','waise',
    'police','train','office','cab','hotel','station','phone','doctor','class',
    'sir','sahab','madam','mr','mrs','miss','dr','lord','lady','aur','main','maine',
    'usne','unhone','tumne','seedha','fir','abhi','aap','aapka','aapki','aapko',
}

def has_devanagari(text): return bool(re.search(r'[\u0900-\u097F]', text))

def extract_plot_note(text):
    marker = '##PLOT_NOTE:'
    if marker in text:
        parts = text.split(marker, 1)
        note_line = parts[1].strip().split('\n')[0].strip()
        note_line = re.sub(r'^[\[\(\*]+|[\]\)\*]+$', '', note_line).strip()[:250]
        return parts[0].strip(), note_line
    return text, ''

_COMMENTARY_STARTS = (
    'Yaar, yeh sab','Yaar, woh','Matlab, woh','Matlab, yeh',
    'Obviously, woh','Basically, woh','Seedha bolun toh','Seedha bolu toh',
    'Woh toh seedha','Woh toh bas yeh',
)

def _strip_commentary_paragraphs(text):
    paras = text.split('\n\n')
    clean = []
    for p in paras:
        s = p.strip()
        if not s or s in ('...','—','–','…'): continue
        if any(s.startswith(m) for m in _COMMENTARY_STARTS): continue
        words = s.split()
        if len(words) < 50 and not ('"' in s or '\u201c' in s or '\u201d' in s):
            meta_c = sum(1 for m in ('Yaar','Matlab','Obviously','Basically','Seedha') if m in s)
            if meta_c >= 2: continue
        clean.append(p)
    return '\n\n'.join(clean)

def _break_long_sentences(text, max_words=45):
    break_words = {'aur','lekin','magar','kyunki','isliye','toh','phir','jabki','halanki','tab','jab'}
    result = []
    for para in text.split('\n'):
        words = para.split()
        if len(words) <= max_words:
            result.append(para); continue
        new_para, current = [], []
        for idx, w in enumerate(words):
            current.append(w)
            w_clean = w.lower().strip('.,;:!?"\'')
            if (len(current) >= max_words // 2 and
                    w_clean in break_words and
                    idx < len(words) - 3):
                new_para.append(' '.join(current)); current = []
        if current: new_para.append(' '.join(current))
        result.append('\n'.join(new_para))
    return '\n'.join(result)

def clean_translation(text):
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = re.sub(r'</?think>', '', text)
    text = re.sub(r'[\u0900-\u097F]+', '', text)
    text = re.sub(r'  +', ' ', text)
    text = _strip_german_residuals(text)
    text = re.sub(r'\bwohne\b', 'usne', text)
    text = re.sub(r'\bletah\b', 'leta', text)
    text = re.sub(r'\bwoh ne\b', 'usne', text)
    text = re.sub(r'```\w*\n?', '', text)
    text = re.sub(r'```', '', text)
    text = re.sub(r'^(Translation:|Hindi Translation:|Hinglish Translation:|Here.s the translation:|Hinglish:|Hindi:)\s*',
                  '', text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r'=== STORY CONTEXT.*?=== CONTEXT END.*?===', '', text, flags=re.DOTALL)
    text = re.sub(r'\[PICHLI TRANSLATION.*?\[YAHAN SE NAYI.*?\]', '', text, flags=re.DOTALL)
    text = re.sub(r'PICHLI TRANSLATION KA.*?\n', '', text, flags=re.DOTALL)
    text = re.sub(r'[\u2500-\u257F]{3,}', '', text)
    text = re.sub(r'(?m)^[=\-\*]{5,}\s*$', '', text)
    text = re.sub(r'(?m)^---\s*$', '', text)
    text = _strip_commentary_paragraphs(text)
    text = _break_long_sentences(text)
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        s = line.strip()
        if s: cleaned.append(s)
        elif cleaned and cleaned[-1] != '': cleaned.append('')
    text = '\n'.join(cleaned)
    return re.sub(r'\n{3,}', '\n\n', text).strip()

def validate_translation(translated, prev_tail='', chunk_index=0, orig_text=''):
    issues = {}
    deva = re.findall(r'[\u0900-\u097F]+', translated)
    issues['devanagari'] = (False, f'{len(deva)} segment(s)') if deva else (True, 'Clean')
    seps = re.findall(r'[\u2500-\u257F]{3,}', translated)
    seps += re.findall(r'(?m)^[=\-\*]{5,}\s*$', translated)
    issues['separators'] = (False, f'{len(seps)}') if seps else (True, 'Clean')
    eng_runs = re.findall(r'\b[A-Za-z]{3,}(?:\s+[A-Za-z]{3,}){4,}\b', translated)
    real = [r for r in eng_runs
            if sum(1 for w in r.split() if re.match(r'^[a-z]{4,}$',w) and w not in _H_ALLOW) >= 3]
    issues['untranslated'] = (False, f'{len(real)} run(s)') if real else (True, 'Clean')
    if prev_tail and len(prev_tail) > 20:
        pw = set(w.lower() for w in prev_tail.split()[-15:])
        cw = set(w.lower() for w in translated.split()[:15])
        sim = len(pw & cw) / max(len(pw), 1)
        issues['overlap_dup'] = (False, f'High sim {sim:.0%}') if sim > 0.6 else (True, f'OK {sim:.0%}')
    else:
        issues['overlap_dup'] = (True, 'n/a')
    # Anti-hallucination: check length ratio
    if orig_text:
        src_w = len(orig_text.split())
        out_w = len(translated.split())
        ratio = out_w / max(src_w, 1)
        if ratio > 2.5:
            issues['hallucination_risk'] = (False, f'Output {ratio:.1f}x longer than source — possible fabrication')
        elif ratio < 0.4:
            issues['hallucination_risk'] = (False, f'Output only {ratio:.1f}x source — possible truncation')
        else:
            issues['hallucination_risk'] = (True, f'Ratio {ratio:.1f}x OK')
    return issues

def summarise_issues(issues_dict):
    lines, ok = [], True
    for check, (passed, detail) in issues_dict.items():
        if not passed: ok = False
        lines.append(f'  {"✅" if passed else "⚠️ "} {check}: {detail}')
    return '\n'.join(lines), ok


# ═══════════════════════════════════════════════════════════════════
# BACKGROUND FILE WRITER (v10.0 — no translation blocking)
# ═══════════════════════════════════════════════════════════════════
class BackgroundWriter:
    """Writes translation chunks to file in background thread — no blocking."""
    def __init__(self, filepath):
        self.filepath = filepath
        self._queue = []
        self._lock = threading.Lock()
        self._thread = None

    def write_chunk(self, text):
        with self._lock:
            self._queue.append(text)
        t = threading.Thread(target=self._flush, daemon=True)
        t.start()

    def _flush(self):
        with self._lock:
            if not self._queue: return
            items = list(self._queue)
            self._queue.clear()
        try:
            with open(self.filepath, 'a', encoding='utf-8') as f:
                for item in items:
                    f.write(item + '\n\n')
        except Exception as e:
            print(f'[BG WRITE ERROR] {e}')

    def finalize(self):
        """Ensure all queued items are written."""
        self._flush()


# ═══════════════════════════════════════════════════════════════════
# OLLAMA TRANSLATION ENGINE v10.0
# ═══════════════════════════════════════════════════════════════════
class OllamaTranslationEngine:
    """
    2-Pass Hinglish Translation Engine v10.0
    Pass 1: Dead-boring faithful Hindi base (temp=0.15)
    Pass 2: Hinglish style filter with ToneGuard Pro + 8 few-shots (temp=0.65)
    """

    _OPTS_STEP1 = {
        'temperature': 0.15, 'top_k': 20, 'top_p': 0.85,
        'repeat_penalty': 1.05, 'num_predict': 1500, 'num_ctx': 8192,
    }
    _OPTS_STEP2 = {
        'temperature': 0.65, 'top_k': 40, 'top_p': 0.90,
        'repeat_penalty': 1.10, 'num_predict': 2048, 'num_ctx': 8192,
    }

    def __init__(self, model_name, target_lang, tier='ADVANCED', num_ctx=8192, tone_rules=None):
        self.model_name  = model_name
        self.target_lang = target_lang
        self.tier        = tier
        self.lang_name   = LANG_NAMES.get(target_lang, target_lang)
        self.tone_rules  = tone_rules or {}
        self._OPTS_STEP1 = dict(self._OPTS_STEP1); self._OPTS_STEP1['num_ctx'] = num_ctx
        self._OPTS_STEP2 = dict(self._OPTS_STEP2); self._OPTS_STEP2['num_ctx'] = num_ctx
        arch = '2-Pass' if tier == 'ADVANCED' else 'Single-Pass'
        print(f'📥 Engine init: {model_name} | {self.lang_name} | {tier} ({arch})')
        print(f'   Step1: temp=0.15 top_k=20 (faithful base)')
        print(f'   Step2: temp=0.65 top_k=40 (Hinglish + ToneGuard Pro + 8 few-shots)')
        try:
            import ollama
            self.client = ollama
            models = ollama.list()
            available = [m.get('name','').split(':')[0] for m in models.get('models',[])]
            base = model_name.split(':')[0]
            if not any(base in m for m in available):
                print(f'⚠️ {model_name} not found — pulling...')
                ollama.pull(model_name)
            else:
                print(f'✅ {model_name} ready!')
        except Exception as e:
            print(f'❌ Init error: {e}'); raise

    def _build_tone_rules_section(self):
        tr = self.tone_rules
        if not tr: return ''
        lines = ['=== BOOK TONE RULES ===']
        if tr.get('default'): lines.append(f'Default tone: {tr["default"]}')
        if tr.get('avoid'):
            lines.append(f'AVOID: {(", ".join(tr["avoid"]) if isinstance(tr["avoid"],list) else tr["avoid"])}')
        if tr.get('extra'): lines.append(f'Note: {tr["extra"]}')
        lines.append('')
        return '\n'.join(lines)

    def _build_char_styles_section(self):
        styles = STORY_STATE.get('character_speech_styles', {})
        chars  = STORY_STATE.get('characters', {})
        if not chars:
            return '(No characters defined — use neutral default)'
        lines = ['Character styles (FIXED — DO NOT modify):']
        for name, info in list(chars.items())[:10]:
            style = styles.get(name, 'natural — keep neutral')
            addr  = info.get('address', 'aap')
            lines.append(f'  {name} → {style} | address: {addr} — DO NOT change')
        lines.append('Violating any character style = FIDELITY FAIL')
        return '\n'.join(lines)

    def _call_model(self, system, user, options):
        response = self.client.chat(
            model   = self.model_name,
            messages= [
                {'role': 'system', 'content': system},
                {'role': 'user',   'content': user},
            ],
            options = options,
        )
        return response['message']['content']

    def translate(self, text, tier=None, prev_translation='', overlap_words=80, context_block=''):
        t = tier or self.tier
        if t == 'ADVANCED':
            return self._translate_two_pass(text, prev_translation, overlap_words, context_block)
        else:
            return self._translate_single_pass(text, t, prev_translation, overlap_words, context_block)

    def _translate_single_pass(self, text, tier, prev_translation, overlap_words, context_block):
        prompts    = TRANSLATION_PROMPTS[tier]
        scene_type = detect_scene_type(text)
        scene_ctx  = SCENE_CTX.get(scene_type, SCENE_CTX['DAILY_LIFE'])
        overlap    = get_overlap(prev_translation, overlap_words)
        ovlp_sec   = build_overlap_section(overlap)
        user = prompts['user'].format(
            context_block=context_block, scene_context=scene_ctx,
            overlap_section=ovlp_sec, chunk=text)
        for attempt in range(3):
            try:
                raw = self._call_model(prompts['system'], user, self._OPTS_STEP2)
                raw, pn = extract_plot_note(raw)
                translation = clean_translation(raw)
                if has_devanagari(translation) and attempt < 2:
                    print(f'   ⚠️ Devanagari attempt {attempt+1}, retry...'); continue
                return translation, scene_type, pn
            except Exception as e:
                if attempt == 2: raise
                print(f'   ⚠️ Attempt {attempt+1}: {e}')
        return '', scene_type, ''

    def _translate_two_pass(self, text, prev_translation, overlap_words, context_block):
        scene_type = detect_scene_type(text)
        scene_ctx  = SCENE_CTX.get(scene_type, SCENE_CTX['DAILY_LIFE'])
        overlap    = get_overlap(prev_translation, overlap_words)
        ovlp_sec   = build_overlap_section(overlap)
        plot_note_en = ''

        # ── STEP 1: Semantic Base ────────────────────────────────────
        step1_user = STEP1_USER.format(chunk=text)
        base_translation = ''
        for attempt in range(3):
            try:
                raw = self._call_model(STEP1_SYSTEM, step1_user, self._OPTS_STEP1)
                raw = clean_translation(raw)
                if has_devanagari(raw) and attempt < 2:
                    print(f'   ⚠️ Step1 Devanagari attempt {attempt+1}'); continue
                base_translation = raw; break
            except Exception as e:
                if attempt == 2:
                    print(f'   ⚠️ Step1 failed: {e} — using English fallback')
                    base_translation = text
                else:
                    print(f'   ⚠️ Step1 attempt {attempt+1}: {e}')

        # ── STEP 2: Hinglish Style Filter ────────────────────────────
        tone_rules_section = self._build_tone_rules_section()
        char_styles        = self._build_char_styles_section()
        # Author DNA in system if available
        author_dna = STORY_STATE.get('author_dna', '')
        voice_guide_section = _HINGLISH_VOICE_GUIDE
        if author_dna:
            voice_guide_section += f'\n\nAUTHOR-SPECIFIC NOTE: {author_dna[:400]}'
        step2_system = STEP2_SYSTEM.format(
            hinglish_voice_guide=voice_guide_section,
            tone_rules_section=tone_rules_section,
            char_styles=char_styles,
        )
        step2_user = STEP2_USER.format(
            context_block    = context_block,
            scene_context    = scene_ctx,
            few_shots        = _FEW_SHOTS,
            char_styles      = char_styles,
            overlap_section  = ovlp_sec,
            base_translation = base_translation,
        )
        final_translation = ''
        for attempt in range(3):
            try:
                raw = self._call_model(step2_system, step2_user, self._OPTS_STEP2)
                raw, pn = extract_plot_note(raw)
                if pn: plot_note_en = pn
                translation = clean_translation(raw)
                needs_retry = False
                if has_devanagari(translation):
                    print(f'   ⚠️ Step2 Devanagari (attempt {attempt+1})')
                    needs_retry = True
                if not needs_retry:
                    words = translation.split()
                    eng_c = sum(1 for w in words if re.match(r'^[a-z]{4,}$',w) and w not in _H_ALLOW)
                    if eng_c / max(len(words),1) > 0.55:
                        print(f'   ⚠️ Step2 high English ratio (attempt {attempt+1})')
                        needs_retry = True
                if not needs_retry or attempt == 2:
                    final_translation = translation; break
            except Exception as e:
                if attempt == 2:
                    print(f'   ⚠️ Step2 failed: {e} — using Step1 base')
                    final_translation = base_translation
                else:
                    print(f'   ⚠️ Step2 attempt {attempt+1}: {e}')

        return final_translation, scene_type, plot_note_en


# ═══════════════════════════════════════════════════════════════════
# TRANSLATION GENERATOR v10.0
# ═══════════════════════════════════════════════════════════════════
class OllamaTranslationGenerator:
    def __init__(self, model_name, target_lang, output_dir='.',
                 tier='ADVANCED', chunk_size=350, overlap_words=80,
                 num_ctx=8192, tone_rules=None, session_budget=None):
        self.model_name    = model_name
        self.target_lang   = target_lang
        self.output_dir    = Path(output_dir)
        self.tier          = tier
        self.chunk_size    = chunk_size
        self.overlap_words = overlap_words
        self.session_budget = session_budget  # seconds
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.engine = OllamaTranslationEngine(
            model_name, target_lang, tier, num_ctx, tone_rules=tone_rules
        )

    def translate_file(self, input_file, resume_from_chunk=0):
        arch = '2-Pass' if self.tier == 'ADVANCED' else 'Single-Pass'
        _jarvis_header(
            f'TRANSLATION GENERATOR v10.0 — {arch} | AuthorDNA | ToneGuard Pro',
            f'Model: {self.model_name} | Tier: {self.tier} | Chunk: {self.chunk_size}w | Session Guard: {self.session_budget//60 if self.session_budget else "OFF"}min')
        with open(input_file, 'r', encoding='utf-8') as f:
            text = f.read()
        text = _preprocess_english_source(text)
        text = clean_source_text(text)
        orig_chars = len(text)
        _jarvis_info(f'📊 Input: {len(text.split()):,} words | {orig_chars:,} chars')
        chunks = chunk_text(text, self.chunk_size)
        STORY_STATE['total_chunks'] = len(chunks)
        est_lo = len(chunks) * 90 // 60
        est_hi = len(chunks) * 120 // 60
        _jarvis_ok(f'📦 {len(chunks)} chunks ({self.chunk_size}w) | Est: {est_lo}–{est_hi} min | Resume from: {resume_from_chunk}')

        # Author DNA notification
        if STORY_STATE.get('author_dna'):
            _jarvis_ok(f'🧬 AuthorDNA: {STORY_STATE["author"]} | Tone: {STORY_STATE["tone_anchor"]}')

        translations = []
        deva_flags   = []
        start_time   = time.time()
        session_start = start_time
        _ctimes, _errs, _ewma = [], 0, None
        _ALPHA = 0.25
        qa_report  = []
        timestamp  = datetime.now().strftime('%Y%m%d_%H%M%S')
        lang_code  = self.target_lang.split('_')[0]
        state_file = self.output_dir / f'state_{lang_code}_{timestamp}.json'
        out_path   = self.output_dir / f'translation_{lang_code}_{timestamp}.txt'

        # Background writer — no translation blocking
        bg_writer = BackgroundWriter(str(out_path))

        for i, chunk in enumerate(chunks, 1):
            # Skip already-translated chunks (resume)
            if i <= resume_from_chunk:
                _jarvis_info(f'[RESUME] Skipping chunk {i} (already done)')
                continue

            # T4 Session Guard — warn if < 30 min remains
            session_elapsed = time.time() - session_start
            if self.session_budget:
                session_remaining = self.session_budget - session_elapsed
                if session_remaining < 600:  # < 10 min
                    _jarvis_warn(f'⚠️ SESSION GUARD: Only {int(session_remaining//60)}m remaining — stopping to preserve output!')
                    _jarvis_warn('Download output now before session expires!')
                    break
                elif session_remaining < 1800 and _ewma and _ewma > session_remaining / max(len(chunks) - i, 1):
                    _jarvis_warn(f'⚠️ Session Guard: ETA exceeds remaining time. Auto-shrinking chunk size.')
                    self.chunk_size = max(150, self.chunk_size - 50)

            chunk_start = time.time()
            prev_trans  = translations[-1] if translations else ''
            prev_tail   = get_overlap(prev_trans, self.overlap_words) if prev_trans else ''
            context_block = build_context_prompt()
            context_chars = len(context_block)

            try:
                translated, scene_type, plot_note_en = self.engine.translate(
                    chunk, self.tier, prev_trans, self.overlap_words, context_block)
                translated = clean_translation(translated)
                issues    = validate_translation(translated, prev_tail, i, chunk)
                qa_sum, all_ok = summarise_issues(issues)
                qa_report.append({'chunk': i, 'scene': scene_type, 'issues': issues})
                deva_flags.append(not issues['devanagari'][0])
                translations.append(translated)

                # Background write — non-blocking
                bg_writer.write_chunk(translated)
                update_story_state(chunk, translated, i, scene_type, plot_note_en=plot_note_en)
                # State save also background
                t_save = threading.Thread(target=save_state, args=(str(state_file),), daemon=True)
                t_save.start()

                chunk_time = time.time() - chunk_start
                _ctimes.append(chunk_time)
                _ewma = chunk_time if _ewma is None else _ALPHA*chunk_time + (1-_ALPHA)*_ewma
                _eta  = _ewma * (len(chunks) - i)
                total_e = time.time() - start_time
                gpu_u, vr_u, vr_t, gpu_t = _j_gpu()
                qa_txt = '' if all_ok else qa_sum.replace('\n',' | ')[:120]
                clear_output(wait=True)
                _jarvis_chunk_dashboard(
                    i, len(chunks), scene_type,
                    len(chunk.split()), len(chunk), len(translated),
                    chunk_time, total_e, _eta, _ctimes, _errs,
                    gpu_u, vr_u, vr_t, gpu_t,
                    self.model_name, self.target_lang, self.tier, all_ok, qa_txt, context_chars,
                    pass_label='2-Pass' if self.tier=='ADVANCED' else '1-Pass',
                    session_budget=self.session_budget,
                    session_elapsed=session_elapsed)

            except Exception as e:
                _errs += 1
                chunk_time = time.time() - chunk_start
                _ctimes.append(chunk_time)
                _ewma = chunk_time if _ewma is None else _ALPHA*chunk_time + (1-_ALPHA)*_ewma
                _eta  = _ewma * (len(chunks) - i)
                total_e = time.time() - start_time
                gpu_u, vr_u, vr_t, gpu_t = _j_gpu()
                err_text = f'[ERROR chunk {i}: {e}]'
                translations.append(err_text)
                bg_writer.write_chunk(err_text)
                deva_flags.append(False)
                qa_report.append({'chunk':i,'scene':'ERROR','issues':{'error':(False,str(e))}})
                clear_output(wait=True)
                _jarvis_chunk_dashboard(
                    i, len(chunks), 'ERROR',
                    len(chunk.split()), len(chunk), 0,
                    chunk_time, total_e, _eta, _ctimes, _errs,
                    gpu_u, vr_u, vr_t, gpu_t,
                    self.model_name, self.target_lang, self.tier, False, f'ERROR: {e}', 0,
                    session_budget=self.session_budget,
                    session_elapsed=time.time()-session_start)

        # Finalize
        bg_writer.finalize()
        total_time  = time.time() - start_time
        trans_chars = sum(len(t) for t in translations)
        deva_count  = sum(deva_flags)
        qa_report_file = self.output_dir / f'qa_{lang_code}_{timestamp}.txt'
        qa_failed_chunks = []
        with open(qa_report_file, 'w', encoding='utf-8') as qf:
            qf.write(f'QA REPORT v10.0 — {timestamp}\nModel: {self.model_name}\n'+'='*60+'\n\n')
            for entry in qa_report:
                iss    = entry['issues']
                failed = [k for k,(p,_) in iss.items() if not p]
                status = 'PASS' if not failed else ('FAIL ['+ ', '.join(failed) +']')
                qf.write(f'Chunk {entry["chunk"]} ({entry["scene"]}): {status}\n')
                if failed:
                    qa_failed_chunks.append(entry['chunk'])
                    for k in failed: qf.write(f'  -> {k}: {iss[k][1]}\n')
            qf.write(f'\nSUMMARY: {len(qa_failed_chunks)}/{len(chunks)} chunks had issues\n')

        _jarvis_complete(self.model_name, total_time, len(translations), orig_chars,
                         trans_chars, deva_count, len(qa_failed_chunks),
                         out_path, str(state_file))
        self.last_state_file = str(state_file)
        self.last_qa_file    = str(qa_report_file)
        return str(out_path), str(state_file)


_jarvis_ok('Translation Engine v10.0 loaded — 2-Pass | AuthorDNA | ToneGuard Pro | Anti-Hallucination')
_jarvis_ok('Step1: Dead-boring base (temp=0.15) | Step2: Hinglish style filter (temp=0.65)')
_jarvis_ok('8 Genre Few-Shots: Kafka · Austen · Doyle · Tagore · Stoker · Action · Emotional · Inner-Monologue')
_jarvis_info('Zero-Fabrication Rule: Output length ratio check | No invented content')
_jarvis_info('T4 Session Guard: Timeout warning | Adaptive chunk shrink | Background file I/O')
_jarvis_info('Context v5.0: rolling 3-chunk summary | capped at 800 chars | author_dna locked')

## ⚡ Step 7 — Configure Book Profile & Run

**You only edit `BOOK_PROFILE` — one dict for the whole book.**

| Field | Description |
|-------|-------------|
| `title` | Book title |
| `author_key` | Key from `AUTHOR_DNA` dict (e.g. `'kafka'`, `'austen'`, `'doyle'`) or `''` for manual |
| `genre` | Override genre (leave `''` to auto-detect or use AuthorDNA default) |
| `characters` | `(name, role, notes, address, speech_style)` — 5-tuple |
| `extra_vocab` | `{english: hinglish}` — fixed translation pairs |
| `tone_rules` | `{default, avoid, extra}` — book-level register |
| `resume_from` | `''` (fresh) · `'auto'` (latest state) · `'path/to/state.json'` |

**Available AuthorDNA keys:** `kafka · austen · doyle · tagore · stoker · tolstoy · dostoevsky · chekhov · flaubert · hugo · maupassant · soseki`


In [9]:
from IPython.display import display, HTML

# ════════════════════════════════════════════════════════════════════════
#   BOOK PROFILE  ←  ONLY THIS SECTION NEEDS TO BE EDITED, ONCE PER BOOK
#
#   HOW TO USE:
#   1. Set 'author_key' to a key from AUTHOR_DNA (e.g. 'kafka', 'austen')
#      → This auto-loads genre, vocab, tone rules, and author voice
#   2. Add your 'title', 'characters', 'extra_vocab'
#   3. 'tone_rules' overrides AuthorDNA defaults if set
#   4. 'resume_from': '' = fresh start | 'auto' = continue last session
#
#   EXAMPLE PROFILES below — uncomment the one you need, or write your own.
# ════════════════════════════════════════════════════════════════════════

BOOK_PROFILE = {

    # ─── BOOK IDENTITY ────────────────────────────────────────────────
    'title': 'The Metamorphosis — Chapter 2',

    # ─── AUTHOR DNA KEY ───────────────────────────────────────────────
    'author_key': 'kafka',

    # ─── GENRE OVERRIDE ───────────────────────────────────────────────
    'genre': '',

    # ─── CHARACTERS ───────────────────────────────────────────────────
    # Format: (name, role, notes, address_form, speech_style)
    'characters': [
        (
            'Gregor',
            'transformed protagonist',
            (
                'Now fully adapted to insect body — crawls walls and ceiling for pleasure. '
                'Hides under sofa covered by a sheet he arranged himself. '
                'Protects the framed picture of the lady in furs like a last human possession. '
                'Food preferences have reversed — likes rotting scraps, hates fresh food. '
                'An apple lodged in his back by Father — festering wound, eyesight failing. '
                'Listens obsessively at doors; internalizes the family\'s financial panic. '
                'Thinks humanly but accepts his situation with flat, bureaucratic resignation.'
            ),
            'woh',
            'anxious internal monologue, long run-on sentences, self-rationalizing — never dramatic'
        ),
        (
            'Grete',
            'younger sister turned caretaker',
            (
                'Has made herself the family expert on Gregor\'s needs. '
                'Cleans room daily, brings food on old newspaper, tests his taste. '
                'Wants to strip his room bare so he can crawl freely — mother disagrees. '
                'Acts with childish authority and a flair for the dramatic. '
                'First time she speaks directly to Gregor: "You, Gregor!" — shaking fist. '
                'Still caring, but increasingly cold and efficient.'
            ),
            'tum',
            'practical, matter-of-fact with family; brief and cold when addressing Gregor directly'
        ),
        (
            'Father',
            'head of household — newly authoritative',
            (
                'Now wears a tight blue bank uniform with gold buttons every day — even at home. '
                'Double chin above stiff collar; white hair meticulously combed. '
                'Has a job at a bank — family\'s financial lifeline. '
                'Bombards Gregor with apples from the fruit bowl; one lodges in his back. '
                'Misreads every situation — assumes Gregor attacked someone. '
                'Completely changed from the tired, dressing-gown man of before.'
            ),
            'aap',
            'commanding, very short sentences, no explanations — only orders and judgments'
        ),
        (
            'Mother',
            'frail, asthmatic matriarch',
            (
                'Asthmatic — a walk across the apartment exhausts her. '
                'Wanted to see Gregor for weeks; finally enters room to help move furniture. '
                'Argues against clearing the room — wants everything kept exactly as it was. '
                'Faints when she sees Gregor\'s brown shape on the wallpaper. '
                'Runs out in nightgown, skirt falling off, pleading with Father to spare Gregor.'
            ),
            'aap',
            'gentle, whispering, pleading — terrified but fiercely protective'
        ),
    ],

    # ─── EXTRA VOCAB ──────────────────────────────────────────────────
    # AuthorDNA 'kafka' already covers: transformation→tabdeeli, vermin→keeda-makoda,
    # debt→karza, father→Papa, mother→Maa, sister→behen, office→office
    # These are Chapter 2 specific additions:
    'extra_vocab': {
        'father':               'Papa',
        'mother':               'Maa',
        'sister':               'behen',
        'living room':          'baithak',
        'front room':           'aagla kamra',
        'ceiling':              'chhat',
        'sofa':                 'sofa',
        'armchair':             'aaraam kursi',
        'sheet':                'chadar',
        'chest of drawers':     'badi almari',
        'the chest':            'woh almari',
        'desk':                 'desk',
        'sideboard':            'sideboard',
        'fruit bowl':           'phalon ki tokri',
        'apple':                'seb',
        'uniform':              'vardi',
        'gold buttons':         'sone ke button',
        'double chin':          'double chin',
        'boot soles':           'joote ke tale',
        'antennae':             'moochen',
        'the picture':          'woh tasveer',
        'lady in furs':         'khaal wale kapdon wali aurat',
        'wallpaper':            'deewaar ka kaagaz',
        'nightgown':            'raat ka libaas',
        'tonic':                'dawai',
        'corrosive medicine':   'tez dawai',
        'asthma':               'dama',
        'safe':                 'tijori',
        'guilders':             'thodi si raqam',
        'conservatory':         'music school',
        'commission':           'commission',
        'traveling salesman':   'traveling salesman',
        'business':             'business',
        'debt':                 'karza',
        'newspaper':            'newspaper',
        'the maid':             'kaam waali bai',
        'the girl':             'kaam waali bai',
        'housekeeper':          'kaam waali bai',
        'Charlottenstrasse':    'Charlottenstrasse',
        'bank':                 'bank',
    },

    # ─── TONE RULES ───────────────────────────────────────────────────
    'tone_rules': {
        'default': 'flat, neutral, matter-of-fact — Kafka register, unchanged from Chapter 1',
        'avoid':   [
            'sarcasm',
            'attitude',
            'street slang',
            'GenZ filler words',
            'dramatic exclamations',
            'horror-genre language',
            'emotional amplification',
        ],
        'extra': (
            'The festering apple, the failing eyesight, the family\'s financial panic — '
            'all described with the same dead-eyed flatness as a grocery list. '
            'Gregor\'s slow death is treated as a logistical inconvenience, not a tragedy. '
            'Father\'s apple-throwing is mechanical, not monstrous. '
            'Mother\'s nightgown collapse is ordinary, not operatic.'
        ),
    },

    # ─── RESUME ───────────────────────────────────────────────────────
    # 'auto' = picks up from where Chapter 1 translation ended
    'resume_from': 'auto',

}


# ════════════════════════════════════════════════════════════════════════
#   MORE BOOK PROFILE EXAMPLES — uncomment & replace BOOK_PROFILE above
# ════════════════════════════════════════════════════════════════════════

# ── Pride and Prejudice (Austen) ──────────────────────────────────────
# BOOK_PROFILE = {
#     'title': 'Pride and Prejudice',
#     'author_key': 'austen',
#     'genre': '',
#     'characters': [
#         ('Elizabeth', 'witty protagonist', 'Sharp, observant, refuses to be bought', 'tum', 'witty, ironic, sharp'),
#         ('Mr. Darcy',  'wealthy love interest', 'Proud, reserved, secretly kind', 'aap', 'formal, measured, cold then warm'),
#         ('Jane',       'elder sister', 'Sweet, trusting, optimistic', 'tum', 'gentle, warm, charitable'),
#         ('Mrs. Bennet','anxious mother', 'Marriage-obsessed, dramatic', 'aap', 'anxious, voluble, excitable'),
#         ('Mr. Bennet', 'sardonic father', 'Witty, detached, disappointed', 'aap', 'dry, ironic, detached'),
#         ('Mr. Bingley','amiable gentleman', 'Friendly, earnest, easily influenced', 'aap', 'warm, enthusiastic, uncomplicated'),
#     ],
#     'extra_vocab': {
#         'entail': 'zameen ka kanooni baandhan', 'militia': 'fauj',
#         'five thousand a year': 'kaafi zyada daulat', 'Netherfield Park': 'Netherfield Park',
#         'Longbourn': 'Longbourn', 'Pemberley': 'Pemberley',
#     },
#     'tone_rules': {
#         'default': 'witty, ironic, socially observant — gentle Austen register',
#         'avoid': ['sarcasm that becomes cruelty', 'modern slang', 'street language'],
#         'extra': 'The irony must be in the understatement — never stated loudly.',
#     },
#     'resume_from': '',
# }

# ── Sherlock Holmes — A Study in Scarlet (Doyle) ─────────────────────
# BOOK_PROFILE = {
#     'title': 'A Study in Scarlet',
#     'author_key': 'doyle',
#     'genre': '',
#     'characters': [
#         ('Holmes',  'eccentric detective', 'Analytical, cold, brilliant — never casual', 'aap', 'calm, precise, analytical — cold'),
#         ('Watson',  'narrator and companion', 'Warm, admiring, military man', 'tum', 'warm, earnest, observational'),
#         ('Lestrade','Scotland Yard inspector', 'Conventional, slow, grudgingly respects Holmes', 'aap', 'official, a little pompous'),
#     ],
#     'extra_vocab': {
#         'Baker Street': 'Baker Street', 'Scotland Yard': 'Scotland Yard',
#         'revolver': 'revolver', 'hansom': 'buggy', 'telegraph': 'taar',
#         'constable': 'constable', 'the Yard': 'Scotland Yard',
#     },
#     'tone_rules': {
#         'default': 'confident, precise, Victorian-formal',
#         'avoid': ['casual address to Holmes', 'bhai/yaar', 'modern idioms'],
#         'extra': 'Holmes is always aap. His deductions must feel logical and cold, not dramatic.',
#     },
#     'resume_from': '',
# }

# ── Dracula (Stoker) ──────────────────────────────────────────────────
# BOOK_PROFILE = {
#     'title': 'Dracula',
#     'author_key': 'stoker',
#     'genre': '',
#     'characters': [
#         ('Jonathan Harker', 'young solicitor', 'Rational, brave, increasingly terrified', 'main/Jonathan', 'formal diary-entry voice, increasingly desperate'),
#         ('Count Dracula',   'vampire antagonist', 'Powerful, ancient, unsettling, charming', 'Count sahab', 'ceremonious, cold, formal'),
#         ('Mina',           'protagonist wife', 'Intelligent, brave, compassionate', 'tum', 'warm, composed, perceptive'),
#         ('Van Helsing',    'vampire hunter', 'Wise, determined, broken English', 'aap', 'academic, formal, slightly archaic'),
#     ],
#     'extra_vocab': {
#         'Transylvania': 'Transylvania', 'Castle Dracula': 'Castle Dracula',
#         'wolfsbane': 'ek khas phool', 'crucifex': 'Isa ka nishaan',
#         'journal': 'diary', 'the Count': 'Count sahab',
#     },
#     'tone_rules': {
#         'default': 'formal diary-entry, Victorian, dread-building through description',
#         'avoid': ['dramatic exclamations', 'modern slang', 'casual address'],
#         'extra': 'Horror builds through cold flat description, not through exclamation.',
#     },
#     'resume_from': '',
# }

# ── The Brothers Karamazov / Crime & Punishment (Dostoevsky) ─────────
# BOOK_PROFILE = {
#     'title': 'Crime and Punishment',
#     'author_key': 'dostoevsky',
#     'genre': '',
#     'characters': [
#         ('Raskolnikov', 'tormented student-murderer', 'Brilliant, paranoid, guilt-ridden', 'main/Raskolnikov', 'fragmented, spiraling, self-contradicting'),
#         ('Sonia',       'compassionate young woman',  'Devout, suffering, forgiving', 'tum', 'gentle, sincere, morally certain'),
#         ('Porfiry',     'shrewd investigator',        'Sharp, plays games, lets suspect sweat', 'aap', 'ironic, measured, never reveals hand'),
#         ('Dunya',       'protagonist sister',         'Strong-willed, principled', 'tum', 'proud, firm, caring'),
#     ],
#     'extra_vocab': {
#         'kopek': 'paisa', 'ruble': 'ruble', 'Petersburg': 'Petersburg',
#         'axe': 'kulhaadi', 'pawnbroker': 'sahukar aurat', 'conscience': 'zameer',
#     },
#     'tone_rules': {
#         'default': 'psychologically intense, fragmented, morally anguished',
#         'avoid': ['casual slang', 'flippancy', 'neat resolution of inner conflict'],
#         'extra': 'Inner monologue must feel unstable — characters contradict themselves mid-thought.',
#     },
#     'resume_from': '',
# }

# ── Gitanjali / Gora (Tagore) ────────────────────────────────────────
# BOOK_PROFILE = {
#     'title': 'Gora',
#     'author_key': 'tagore',
#     'genre': '',
#     'characters': [
#         ('Gora',      'passionate protagonist', 'Fierce Hindu nationalist, searching for identity', 'woh/Gora', 'passionate, formal, principled'),
#         ('Sucharita', 'thoughtful young woman', 'Brahmo, calm, intelligent', 'tum', 'composed, gentle, searching'),
#         ('Binoy',     'Gora\'s friend',          'Moderate, loving, torn between worlds', 'tum', 'warm, earnest, doubting'),
#     ],
#     'extra_vocab': {
#         'Brahmo': 'Brahmo Samaj', 'Arya': 'Arya', 'ashram': 'ashram',
#         'namaz': 'namaz', 'puja': 'puja', 'dharma': 'dharma',
#     },
#     'tone_rules': {
#         'default': 'lyrical, philosophical, emotionally elevated',
#         'avoid': ['casual slang', 'reductive summaries of deep emotion', 'flippancy'],
#         'extra': 'Tagore\'s Bengal — philosophical debates must feel real and weighted, not academic.',
#     },
#     'resume_from': '',
# }


# ════════════════════════════════════════════════════════════════════════
#   NOTHING BELOW THIS LINE NEEDS TO CHANGE
# ════════════════════════════════════════════════════════════════════════

reset_for_new_book(title=BOOK_PROFILE['title'], genre=BOOK_PROFILE.get('genre',''))

# Load AuthorDNA if specified
if BOOK_PROFILE.get('author_key'):
    load_author_dna(BOOK_PROFILE['author_key'])

# Override genre if explicitly set
if BOOK_PROFILE.get('genre'):
    set_genre(BOOK_PROFILE['genre'])

# Register characters (5-tuple or 4-tuple)
for char_entry in BOOK_PROFILE['characters']:
    if len(char_entry) == 5:
        name, role, notes, addr, speech_style = char_entry
    elif len(char_entry) == 4:
        name, role, notes, addr = char_entry; speech_style = ''
    else:
        print(f'[WARN] Skipping malformed character: {char_entry}'); continue
    update_character(name, role=role, notes=notes, address=addr,
                     speech_style=speech_style if speech_style else None)

# Load extra vocab (AFTER author DNA so these override)
for eng, hin in BOOK_PROFILE['extra_vocab'].items():
    add_vocab_entry(eng, hin)

# Resume logic
_resume = BOOK_PROFILE['resume_from'].strip()
_resume_chunk = 0
if _resume == 'auto':
    if not _load_latest_state(OUTPUT_DIR):
        print('[INFO] No previous state — starting fresh')
    else:
        _resume_chunk = STORY_STATE.get('resume_from_chunk', STORY_STATE.get('chunk_count', 0))
elif _resume:
    load_state(_resume)
    _resume_chunk = STORY_STATE.get('resume_from_chunk', STORY_STATE.get('chunk_count', 0))

print()
print_state_summary()

# Merge BOOK_PROFILE tone_rules with AuthorDNA (profile wins)
_tone_rules = {}
if BOOK_PROFILE.get('author_key') and BOOK_PROFILE['author_key'] in AUTHOR_DNA:
    _dna_tone, _, _, _ = AUTHOR_DNA[BOOK_PROFILE['author_key']]
    _tone_rules['default'] = _dna_tone
_tone_rules.update({k:v for k,v in BOOK_PROFILE['tone_rules'].items() if v})

display(HTML(f"""
<div style='background:linear-gradient(135deg,#0a0a0f,#1a0505);border:2px solid #c0392b;
            border-radius:8px;padding:16px 20px;font-family:Courier New,monospace;margin:12px 0;'>
  <div style='color:#c0392b;font-size:1.25em;font-weight:bold;letter-spacing:3px;'>JARVIS v10.0 — PIPELINE READY</div>
  <table style='color:#e8e8e8;font-size:0.85em;margin-top:10px;border-collapse:collapse;'>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>MODEL</td><td>{MODEL}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>TIER</td><td>{TRANSLATION_TIER} {'(2-pass)' if TRANSLATION_TIER=='ADVANCED' else '(1-pass)'}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>BOOK</td><td>{BOOK_PROFILE['title'] or 'not set'}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>AUTHOR KEY</td>
        <td style='color:#4CAF50;'>{BOOK_PROFILE.get('author_key','—').upper() or '—'}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>GENRE</td><td>{STORY_STATE['genre'] or 'auto-detect'}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>TONE ANCHOR</td>
        <td style='color:#4CAF50;'>{STORY_STATE.get('tone_anchor','—')}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>CHARACTERS</td><td>{len(BOOK_PROFILE['characters'])}</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>VOCAB</td>
        <td>{len(STORY_STATE['established_vocab'])} entries (AuthorDNA + extra)</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>CHUNK / OVERLAP</td>
        <td>{CHUNK_SIZE}w / {OVERLAP_WORDS}w</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>SESSION BUDGET</td>
        <td style='color:#FFD700;'>{SESSION_BUDGET//60} min</td></tr>
    <tr><td style='color:#FFD700;padding:2px 14px 2px 0;'>RESUME</td>
        <td style='color:{"#4CAF50" if BOOK_PROFILE["resume_from"] else "#888"};'>
          {BOOK_PROFILE["resume_from"] or "fresh start"} {f"(from chunk {_resume_chunk})" if _resume_chunk else ""}</td></tr>
  </table>
</div>
"""))

print('Initializing v10.0 pipeline...')
generator = OllamaTranslationGenerator(
    model_name    = MODEL,
    target_lang   = 'hinglish',
    output_dir    = OUTPUT_DIR,
    tier          = TRANSLATION_TIER,
    chunk_size    = CHUNK_SIZE,
    overlap_words = OVERLAP_WORDS,
    num_ctx       = NUM_CTX,
    tone_rules    = _tone_rules,
    session_budget = SESSION_BUDGET,
)

print('\nStarting translation...')
OUTPUT_FILE, STATE_FILE = generator.translate_file(UPLOADED_FILE, resume_from_chunk=_resume_chunk)

print(f'\nTranslation : {OUTPUT_FILE}')
print(f'State JSON  : {STATE_FILE}')
print()
print_state_summary()

Total Time,45m 26s
Chunks,4
Avg/Chunk,11m 21s
Input,"6,576 chars"
Output,"6,584 chars"
Expansion,1.0x
Devanagari,PASS — Clean
QA Pipeline,WARN 4 chunks
File,translation_hinglish_20260326_135103.txt



Translation : translation_output/translation_hinglish_20260326_135103.txt
State JSON  : translation_output/state_hinglish_20260326_135103.json


=== STORY STATE v5.0 ===
  Book    : The Metamorphosis — Chapter 2
  Author  : Kafka
  Genre   : literary fiction
  Tone    : unsettling, flat, bureaucratic
  Chunks  : 4 / 4
  Chars   : 4 | Speech: 4
  Vocab   : 35 entries
  Last 3  : Gregor's father appears in a new, author → Gregor is forced to run by his father, h → Gregor is attacked with apples by his fa


## ⬇️ Step 8 — Download Outputs
Downloads translation file + state.json for session resume.

In [10]:
from google.colab import files
from IPython.display import display, HTML

display(HTML('<div style="background:#0a0f0a;border:2px solid #4CAF50;border-radius:8px;'
            'padding:12px 18px;font-family:Courier New,monospace;">'
            '<div style="color:#4CAF50;font-size:1.1em;font-weight:bold;">[>] OUTPUT EXTRACTION — JARVIS v10.0</div>'
            '<div style="color:#888;font-size:0.82em;margin-top:4px;">Downloading translation + state.json</div></div>'))

print('📥 Downloading translation...')
files.download(OUTPUT_FILE)
print('📥 Downloading state.json (for resume)...')
files.download(STATE_FILE)

print('\n✅ Both files downloaded!')
print('\n💡 To resume Chapter 2 (new session):')
print('   1. Run all setup cells (1–14)')
print('   2. In BOOK_PROFILE, set resume_from = \'auto\' or paste state.json path')
print('   3. Upload Chapter 2 text and run Cell 16.')
print('   All characters, vocab, author tone, and story context will be restored.')

📥 Downloading translation...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Downloading state.json (for resume)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Both files downloaded!

💡 To resume Chapter 2 (new session):
   1. Run all setup cells (1–14)
   2. In BOOK_PROFILE, set resume_from = 'auto' or paste state.json path
   3. Upload Chapter 2 text and run Cell 16.
   All characters, vocab, author tone, and story context will be restored.


## 💾 (Optional) Step 9 — Save to Google Drive

In [11]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/JARVIS_Translation'
os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copy(OUTPUT_FILE, os.path.join(DRIVE_DIR, os.path.basename(OUTPUT_FILE)))
shutil.copy(STATE_FILE,  os.path.join(DRIVE_DIR, os.path.basename(STATE_FILE)))

print(f'✅ Saved to Google Drive: {DRIVE_DIR}')
print(f'   • {os.path.basename(OUTPUT_FILE)}')
print(f'   • {os.path.basename(STATE_FILE)}')

MessageError: Error: credential propagation was unsuccessful

## 📖 Appendix — Hinglish Voice Reference

This is the researched tone guide that JARVIS uses internally. Read this to understand what 'right Hinglish' sounds like for this project.

### The Target Voice
**Like an educated Indian in their late 20s/early 30s narrating a story to a friend.**  
Reference: *Made in Heaven*, *Little Things*, *Scam 1992* — clear, warm, intelligent.

### What This Sounds Like
- Mostly Hindi/Urdu vocabulary, English nouns where they're actually more natural
- Sentences flow like speech — not textbook Hindi, not pidgin English
- Emotions felt fully — expressed with restraint, not melodrama
- Formal characters speak with dignity — not casually addressed

### Address Forms
| Context | Form |
|---------|------|
| Formal / respectful | `aap / aapko / aapka` |
| Familiar / equal | `tum / tumhe / tumhara / tumne` |
| NEVER in this project | `tu / tujhe / tera / tune` |

### Never Use
- `yaar` or `bhai` as address terms for characters
- `gadha`, `bewakoof`, `ullu` (abusive tone)
- English Gen-Z slang (`no cap`, `based`, `lowkey`)
- Forced exclamations (`!!!`)
- Narrator commentary not in the source

### Author-Specific Notes

| Author | Key Hinglish Principle |
|--------|------------------------|
| **Kafka** | Horror = flatness. Describe transformation like a tax form. |
| **Austen** | Irony lives in understatement. The humor is never announced. |
| **Doyle** | Holmes is always formal. His Hinglish = composed professional. |
| **Tagore** | Lyricism must survive. Sentences breathe. No flippancy. |
| **Stoker** | Dread builds through cold description, not exclamation. |
| **Dostoevsky** | Inner monologue spirals. Characters contradict themselves. |
| **Tolstoy** | Gravitas maintained. No casualness in serious moments. |
| **Chekhov** | Silence between lines matters. Understate everything. |
